# 🎯 Glimpse3D - Master Pipeline

## Complete 2D Image → 3D Gaussian Splat Pipeline

This notebook runs the **entire Glimpse3D pipeline** end-to-end:

```
📷 Input Image
    ↓
🔷 TripoSR (0.5s) → Initial 3D Mesh → Gaussian Points
    ↓
🎨 SyncDreamer (2min) → 16 Consistent Multi-View Images
    ↓  
✨ SDXL Lightning + ControlNet → Enhanced Views
    ↓
🔮 gsplat Optimization → Refined Gaussians
    ↓
🔄 MVCRM → Multi-View Consistent Refinement
    ↓
🏆 Final 3D Gaussian Splat Output
```

## Requirements
- Google Colab with **T4 GPU** (free tier) or **A100** (faster)
- ~12GB VRAM peak usage
- ~30 minutes total runtime

---

## 🚀 Quick Start

1. Run all cells in order (Runtime → Run all)
2. Upload your image when prompted
3. Wait ~30 minutes for full pipeline
4. Download final results!

# Stage 0: Environment Setup

This stage:
1. Checks GPU availability and VRAM
2. Clones the Glimpse3D repository for `ai_modules` utilities
3. Installs all required dependencies
4. Creates the output directory structure

In [ ]:
# Check environment
import sys
import os

IN_COLAB = 'google.colab' in sys.modules
print(f"🖥️ Running in Colab: {IN_COLAB}")

if not IN_COLAB:
    print("⚠️ This notebook is designed for Google Colab!")
    print("   Some features may not work locally.")

# Check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

import torch
print(f"\n📦 PyTorch: {torch.__version__}")
print(f"🔥 CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🔧 CUDA Version: {torch.version.cuda}")

if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_VRAM = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"🎮 GPU: {GPU_NAME}")
    print(f"💾 VRAM: {GPU_VRAM:.1f} GB")
    
    # Set batch sizes based on GPU
    if GPU_VRAM >= 40:  # A100
        BATCH_VIEW_NUM = 8
        MC_RESOLUTION = 384
        NUM_SAMPLES = 200000
    elif GPU_VRAM >= 15:  # T4/V100
        BATCH_VIEW_NUM = 4
        MC_RESOLUTION = 256
        NUM_SAMPLES = 100000
    else:  # Lower VRAM
        BATCH_VIEW_NUM = 2
        MC_RESOLUTION = 192
        NUM_SAMPLES = 50000
    
    print(f"\n⚙️ Settings: batch_view={BATCH_VIEW_NUM}, resolution={MC_RESOLUTION}, samples={NUM_SAMPLES}")
else:
    raise RuntimeError("❌ No GPU available! Enable GPU in Runtime → Change runtime type")

In [ ]:
# Clone Glimpse3D repository (for ai_modules utilities)
GLIMPSE3D_REPO = "/content/Glimpse-3D"

if not os.path.exists(GLIMPSE3D_REPO):
    print("📥 Cloning Glimpse3D repository...")
    !git clone https://github.com/varunaditya27/Glimpse3D.git {GLIMPSE3D_REPO}
else:
    print(f"✅ Glimpse3D repo already exists at {GLIMPSE3D_REPO}")

# Add to Python path for ai_modules
sys.path.insert(0, GLIMPSE3D_REPO)
print(f"✅ Added {GLIMPSE3D_REPO} to Python path")

## 📦 Installing Dependencies

**Strategy:**
1. Install compatible NumPy + scipy first (avoids version conflicts)
2. Install ML frameworks (transformers, diffusers, etc.)
3. Install 3D processing packages (trimesh, rembg, gsplat)
4. Install model-specific packages (CLIP, etc.)

**Expected warnings:** Pip may show version conflicts for opencv/tensorflow/gradio - these are **harmless** because our pipeline doesn't use those pre-installed Colab packages.

In [ ]:
%%capture install_output
# Install all dependencies - carefully pinned for Google Colab compatibility
# Note: Colab (Jan 2026) has NumPy 2.x, PyTorch 2.9+, CUDA 12.6
# We work WITH these defaults, not against them.

print("📦 Installing dependencies (this takes ~5 minutes)...")

# ===============================================
# STRATEGY: Pin NumPy + scipy first to ensure compatibility
# Then install everything else WITHOUT allowing numpy downgrades
# Ignore pip warnings about opencv/tensorflow/gradio - we don't use them!
# ===============================================

# ⚠️ CRITICAL FIX: Install compatible NumPy + scipy FIRST
# NumPy 2.1.x is well-supported and stable
# scipy 1.14+ has good NumPy 2.x support
print("📌 Step 1: Installing NumPy + scipy with compatible versions...")
!pip install "numpy>=2.1,<2.2" "scipy>=1.14" --quiet

# Verify installation succeeded
import numpy as np
import scipy
print(f"✅ NumPy: {np.__version__}, scipy: {scipy.__version__}")

# Core ML packages - pin to stable versions that work with NumPy 2.x
print("📌 Step 2: Installing ML frameworks...")
!pip install transformers>=4.44.0 diffusers>=0.30.0 accelerate huggingface_hub safetensors --quiet
!pip install omegaconf einops pytorch-lightning>=2.0.0 kornia --quiet

# TripoSR dependencies 
# CRITICAL: trimesh>=4.4.0 is required for NumPy 2.0 compatibility (ptp fix)
print("📌 Step 3: Installing 3D processing packages...")
!pip install "trimesh>=4.4.0" rembg[gpu] xatlas plyfile --quiet

# torchmcubes for marching cubes
!pip install git+https://github.com/tatsy/torchmcubes.git --quiet

# gsplat - builds JIT, compatible with Colab's PyTorch
print("📌 Step 4: Installing gsplat...")
!pip install gsplat --quiet

# SyncDreamer dependencies  
# Pin CLIP to specific commit for stability
print("📌 Step 5: Installing SyncDreamer dependencies...")
!pip install git+https://github.com/openai/CLIP.git@a1d071733d7111c9c014f024669f959182114e33 --quiet
!pip install taming-transformers-rom1504 --quiet

# Image processing - use Colab's scikit-image, just ensure imageio
print("📌 Step 6: Installing image processing...")
!pip install imageio imageio-ffmpeg --quiet

# Depth estimation (MiDaS)
!pip install timm --quiet

print("\n✅ All dependencies installed!")
print("⚠️ Pip warnings about opencv/tensorflow/gradio can be ignored - we don't use those packages.")

In [ ]:
# Verify key dependencies are correctly installed
print("🔍 Verifying critical dependencies...\n")

import numpy as np
print(f"✅ NumPy: {np.__version__}")

import torch
print(f"✅ PyTorch: {torch.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")

# Verify scipy is compatible with numpy (this was causing the _center import error)
import scipy
print(f"✅ scipy: {scipy.__version__}")
# Quick test to ensure scipy.sparse works
try:
    import scipy.sparse
    print("   scipy.sparse: ✅ OK")
except ImportError as e:
    raise RuntimeError(f"❌ scipy is NOT compatible with NumPy! Error: {e}")

import trimesh
print(f"✅ trimesh: {trimesh.__version__}")
# Verify trimesh works with NumPy 2.x (ptp fix check)
try:
    test_mesh = trimesh.creation.box()
    _ = test_mesh.bounds  # This uses np.ptp internally in old versions
    print("   NumPy 2.x compatibility: ✅ OK")
except AttributeError as e:
    if 'ptp' in str(e):
        raise RuntimeError("❌ trimesh is NOT compatible with NumPy 2.x! Please install trimesh>=4.4.0")
    raise

import transformers
print(f"✅ transformers: {transformers.__version__}")

import diffusers
print(f"✅ diffusers: {diffusers.__version__}")

# Test rembg (background removal) - this depends on scipy working
try:
    import rembg
    print(f"✅ rembg: installed")
except ImportError as e:
    print(f"❌ rembg import failed: {e}")
    raise

# Test plyfile
try:
    from plyfile import PlyData
    print(f"✅ plyfile: installed")
except ImportError:
    print("❌ plyfile not found!")

print("\n🎉 All critical dependencies verified!")

In [ ]:
# Create directory structure
from pathlib import Path
import gc

WORK_DIR = Path("/content/glimpse3d_pipeline")
WORK_DIR.mkdir(exist_ok=True)

DIRS = {
    'input': WORK_DIR / 'input',
    'triposr': WORK_DIR / 'stage1_triposr',
    'syncdreamer': WORK_DIR / 'stage2_syncdreamer',
    'enhanced': WORK_DIR / 'stage3_enhanced',
    'gsplat': WORK_DIR / 'stage4_gsplat',
    'mvcrm': WORK_DIR / 'stage5_mvcrm',
    'output': WORK_DIR / 'final_output',
}

for name, path in DIRS.items():
    path.mkdir(exist_ok=True)
    print(f"📁 {name}: {path}")

def clear_gpu():
    """Aggressively clear GPU memory between stages."""
    # Multiple gc passes for thorough cleanup
    for _ in range(3):
        gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"🧹 GPU memory cleared. Allocated: {allocated:.2f} GB, Reserved: {reserved:.2f} GB")

def safe_del(obj_name, globals_dict):
    """Safely delete an object if it exists."""
    if obj_name in globals_dict and globals_dict[obj_name] is not None:
        del globals_dict[obj_name]
        
print("\n✅ Directory structure created!")

# Stage 1: Upload Input Image

In [ ]:
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

print("📤 Upload your image (JPG/PNG):")
uploaded = files.upload()

# Save uploaded file
INPUT_FILENAME = list(uploaded.keys())[0]
INPUT_PATH = DIRS['input'] / INPUT_FILENAME

with open(INPUT_PATH, 'wb') as f:
    f.write(list(uploaded.values())[0])

# Display
input_image = Image.open(INPUT_PATH)
plt.figure(figsize=(8, 8))
plt.imshow(input_image)
plt.title(f"Input: {INPUT_FILENAME} ({input_image.size[0]}x{input_image.size[1]})")
plt.axis('off')
plt.show()

print(f"\n✅ Saved to: {INPUT_PATH}")

# Stage 2: TripoSR - Initial 3D Reconstruction

**Input:** Single image  
**Output:** 3D mesh + Gaussian point cloud  
**Time:** ~30 seconds

In [ ]:
# Clone TripoSR
import sys
import os
from pathlib import Path

TRIPOSR_PATH = Path("/content/TripoSR")

if not TRIPOSR_PATH.exists():
    print("📥 Cloning TripoSR...")
    !git clone https://github.com/VAST-AI-Research/TripoSR.git {TRIPOSR_PATH}

sys.path.insert(0, str(TRIPOSR_PATH))
os.chdir(TRIPOSR_PATH)
print(f"✅ TripoSR ready at {TRIPOSR_PATH}")

In [ ]:
import time
import torch
import numpy as np
from PIL import Image
from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground
import rembg

print("\n" + "="*60)
print("🔷 STAGE 2: TripoSR 3D Reconstruction")
print("="*60)

device = "cuda:0"

# Load model
print("\n📥 Loading TripoSR model...")
triposr_model = TSR.from_pretrained(
    "stabilityai/TripoSR",
    config_name="config.yaml",
    weight_name="model.ckpt",
)
triposr_model.renderer.set_chunk_size(8192)
triposr_model.to(device)
print("✅ Model loaded!")

# Preprocess image
print("\n🔧 Preprocessing image...")
input_img = Image.open(INPUT_PATH)
rembg_session = rembg.new_session()
processed_img = remove_background(input_img, rembg_session)
processed_img = resize_foreground(processed_img, 0.85)

# ✅ FIXED: Save RGBA for SyncDreamer, create RGB for TripoSR
# SyncDreamer's prepare_inputs needs RGBA with alpha channel
processed_img.save(DIRS['triposr'] / "processed_input.png")

# Create RGB version for TripoSR (it expects 3 channels)
img_np = np.array(processed_img).astype(np.float32) / 255.0
img_np_rgb = img_np[:, :, :3] * img_np[:, :, 3:4] + (1 - img_np[:, :, 3:4]) * 0.5
processed_img_rgb = Image.fromarray((img_np_rgb * 255.0).astype(np.uint8))
processed_img_rgb.save(DIRS['triposr'] / "processed_input_rgb.png")

# Run inference
print("\n🚀 Running TripoSR...")
start_time = time.time()

# ⚠️ CRITICAL: Use RGB version for TripoSR (it expects 3 channels, not 4)
with torch.no_grad():
    scene_codes = triposr_model([processed_img_rgb], device=device)
    meshes = triposr_model.extract_mesh(scene_codes, has_vertex_color=True, resolution=MC_RESOLUTION)

mesh = meshes[0]
elapsed = time.time() - start_time

print(f"\n✅ Mesh generated in {elapsed:.2f}s")
print(f"   Vertices: {len(mesh.vertices):,}")
print(f"   Faces: {len(mesh.faces):,}")

# Save mesh - OBJ always works, GLB may fail with NumPy compatibility issues
mesh.export(str(DIRS['triposr'] / "mesh.obj"))
print(f"✅ Saved OBJ: {DIRS['triposr'] / 'mesh.obj'}")

try:
    mesh.export(str(DIRS['triposr'] / "mesh.glb"))
    print(f"✅ Saved GLB: {DIRS['triposr'] / 'mesh.glb'}")
except Exception as e:
    print(f"⚠️ GLB export failed (NumPy compatibility): {e}")
    print(f"   Continuing with OBJ format only")

print(f"\n📁 Mesh saved to {DIRS['triposr']}")

In [ ]:
# ============================================================
# Convert Mesh to Gaussian PLY with Robust Initialization
# ============================================================
import gc
import numpy as np
from plyfile import PlyData, PlyElement
from scipy.spatial import cKDTree

def mesh_to_gaussian_ply(mesh, output_path, num_samples=100000, verbose=True):
    """
    Convert mesh to Gaussian Splat format with robust initialization.
    
    ⚠️ CRITICAL: This function sets up initial Gaussian parameters that
    significantly affect optimization convergence. Poor initialization
    leads to black renders or failed optimization.
    
    Args:
        mesh: trimesh mesh object
        output_path: Path to save PLY file
        num_samples: Number of points to sample from mesh
        verbose: Print diagnostic information
    
    Returns:
        dict with initialization statistics
    """
    if verbose:
        print(f"\n🔄 Converting mesh to Gaussian Splats...")
        print(f"   Mesh vertices: {len(mesh.vertices):,}")
        print(f"   Mesh faces: {len(mesh.faces):,}")
        print(f"   Target samples: {num_samples:,}")
    
    # Sample points from mesh surface
    points, face_indices = mesh.sample(num_samples, return_index=True)
    points = points.astype(np.float32)
    
    # ============================================================
    # Extract colors from mesh
    # ============================================================
    if mesh.visual.vertex_colors is not None and len(mesh.visual.vertex_colors) > 0:
        if verbose:
            print(f"   ✅ Using vertex colors from mesh")
        face_vertices = mesh.faces[face_indices]
        vertex_colors = mesh.visual.vertex_colors[:, :3] / 255.0
        # Average colors of face vertices
        colors = vertex_colors[face_vertices].mean(axis=1).astype(np.float32)
    else:
        if verbose:
            print(f"   ⚠️ No vertex colors, using default gray")
        colors = np.ones((num_samples, 3), dtype=np.float32) * 0.6
    
    num_points = len(points)
    xyz = points
    
    # ============================================================
    # Compute optimal Gaussian scale based on point density
    # ============================================================
    if verbose:
        print(f"\n📊 Computing point cloud statistics...")
    
    # Subsample for faster KD-tree computation
    subsample_size = min(10000, num_points)
    subsample_indices = np.random.choice(num_points, subsample_size, replace=False)
    xyz_subsample = xyz[subsample_indices]
    
    tree = cKDTree(xyz_subsample)
    distances, _ = tree.query(xyz_subsample, k=2)  # k=2: self and nearest neighbor
    avg_nn_distance = distances[:, 1].mean()  # Nearest neighbor distance
    median_nn_distance = np.median(distances[:, 1])
    
    if verbose:
        print(f"   Average NN distance: {avg_nn_distance:.6f}")
        print(f"   Median NN distance: {median_nn_distance:.6f}")
    
    # ============================================================
    # Compute initial scale
    # ============================================================
    # ⚠️ CRITICAL: Scale must be appropriate for the model size
    # Too small = invisible Gaussians, Too large = blurry/overlapping
    # Target: Gaussians should slightly overlap (1.5-2x NN distance)
    
    target_scale = median_nn_distance * 1.5
    target_scale = np.clip(target_scale, 0.001, 0.1)  # Clamp to reasonable range
    
    # gsplat uses exp(scale_raw) for positive scales
    scale_raw = np.log(target_scale)
    
    if verbose:
        print(f"   Target Gaussian scale: {target_scale:.6f}")
        print(f"   Scale raw (log): {scale_raw:.4f}")
    
    scales = np.ones((num_points, 3), dtype=np.float32) * scale_raw
    
    # ============================================================
    # Compute initial opacity
    # ============================================================
    # gsplat uses sigmoid(opacity_raw) for [0,1] opacity
    # sigmoid(2.0) ≈ 0.88 - good starting point (visible but not saturated)
    # sigmoid(3.0) ≈ 0.95 - more visible
    
    initial_opacity_raw = 2.5  # sigmoid(2.5) ≈ 0.92
    opacities = np.ones((num_points, 1), dtype=np.float32) * initial_opacity_raw
    
    if verbose:
        print(f"   Initial opacity (sigmoid): {1 / (1 + np.exp(-initial_opacity_raw)):.4f}")
    
    # ============================================================
    # Compute SH DC coefficients from colors
    # ============================================================
    # Color encoding: RGB = 0.5 + C0 * f_dc
    # Therefore: f_dc = (RGB - 0.5) / C0
    
    C0 = 0.28209479177387814  # First spherical harmonic coefficient
    features_dc = ((colors - 0.5) / C0).astype(np.float32)
    features_rest = np.zeros((num_points, 45), dtype=np.float32)
    
    if verbose:
        print(f"   SH DC range: [{features_dc.min():.4f}, {features_dc.max():.4f}]")
    
    # ============================================================
    # Initialize rotations (identity quaternions in wxyz format)
    # ============================================================
    # gsplat expects quaternions in (w, x, y, z) format where w is scalar
    rotations = np.zeros((num_points, 4), dtype=np.float32)
    rotations[:, 0] = 1.0  # w = 1, x = y = z = 0 (identity rotation)
    
    # ============================================================
    # Build PLY data structure
    # ============================================================
    dtype_full = [
        ('x', 'f4'), ('y', 'f4'), ('z', 'f4'),
        ('f_dc_0', 'f4'), ('f_dc_1', 'f4'), ('f_dc_2', 'f4'),
    ]
    for i in range(45):
        dtype_full.append((f'f_rest_{i}', 'f4'))
    dtype_full.extend([
        ('opacity', 'f4'),
        ('scale_0', 'f4'), ('scale_1', 'f4'), ('scale_2', 'f4'),
        ('rot_0', 'f4'), ('rot_1', 'f4'), ('rot_2', 'f4'), ('rot_3', 'f4'),
    ])
    
    elements = np.zeros(num_points, dtype=dtype_full)
    elements['x'] = xyz[:, 0]
    elements['y'] = xyz[:, 1]
    elements['z'] = xyz[:, 2]
    elements['f_dc_0'] = features_dc[:, 0]
    elements['f_dc_1'] = features_dc[:, 1]
    elements['f_dc_2'] = features_dc[:, 2]
    for i in range(45):
        elements[f'f_rest_{i}'] = features_rest[:, i]
    elements['opacity'] = opacities[:, 0]
    elements['scale_0'] = scales[:, 0]
    elements['scale_1'] = scales[:, 1]
    elements['scale_2'] = scales[:, 2]
    elements['rot_0'] = rotations[:, 0]  # w
    elements['rot_1'] = rotations[:, 1]  # x
    elements['rot_2'] = rotations[:, 2]  # y
    elements['rot_3'] = rotations[:, 3]  # z
    
    # Save PLY
    el = PlyElement.describe(elements, 'vertex')
    PlyData([el]).write(str(output_path))
    
    if verbose:
        print(f"\n✅ Saved Gaussian PLY: {output_path}")
        print(f"   Total Gaussians: {num_points:,}")
        file_size_mb = Path(output_path).stat().st_size / 1024 / 1024
        print(f"   File size: {file_size_mb:.2f} MB")
    
    # Return statistics for validation
    return {
        'num_points': num_points,
        'avg_nn_distance': avg_nn_distance,
        'target_scale': target_scale,
        'initial_opacity': 1 / (1 + np.exp(-initial_opacity_raw)),
        'color_range': (colors.min(), colors.max()),
        'xyz_center': xyz.mean(axis=0),
        'xyz_extent': xyz.max() - xyz.min(),
    }


# ============================================================
# Convert the mesh to Gaussian PLY
# ============================================================
INITIAL_PLY_PATH = DIRS['triposr'] / "initial_gaussians.ply"

init_stats = mesh_to_gaussian_ply(
    mesh, 
    INITIAL_PLY_PATH, 
    num_samples=NUM_SAMPLES,
    verbose=True
)

# Display initialization summary
print(f"\n📊 Gaussian Initialization Summary:")
print(f"   Points: {init_stats['num_points']:,}")
print(f"   Scale: {init_stats['target_scale']:.6f}")
print(f"   Opacity: {init_stats['initial_opacity']:.4f}")
print(f"   Color range: [{init_stats['color_range'][0]:.4f}, {init_stats['color_range'][1]:.4f}]")
print(f"   Center: ({init_stats['xyz_center'][0]:.3f}, {init_stats['xyz_center'][1]:.3f}, {init_stats['xyz_center'][2]:.3f})")
print(f"   Extent: {init_stats['xyz_extent'].max():.3f}")

# Cleanup mesh to free memory
del mesh
gc.collect()
torch.cuda.empty_cache()
print("\n🧹 Mesh cleaned up from memory")

# Stage 3: SyncDreamer - Multi-View Generation

**Input:** Processed image  
**Output:** 16 consistent multi-view images  
**Time:** ~2-3 minutes

In [ ]:
print("\n" + "="*60)
print("🎨 STAGE 3: SyncDreamer Multi-View Generation")
print("="*60)

# Clone SyncDreamer
SYNCDREAMER_PATH = Path("/content/SyncDreamer")

if not SYNCDREAMER_PATH.exists():
    print("📥 Cloning SyncDreamer...")
    !git clone https://github.com/liuyuan-pal/SyncDreamer.git {SYNCDREAMER_PATH}

# Download checkpoints
CKPT_DIR = SYNCDREAMER_PATH / "ckpt"
CKPT_DIR.mkdir(exist_ok=True)

!apt -y install -qq aria2

CHECKPOINTS = {
    "syncdreamer-pretrain.ckpt": "https://huggingface.co/camenduru/SyncDreamer/resolve/main/syncdreamer-pretrain.ckpt",
    "ViT-L-14.pt": "https://huggingface.co/camenduru/SyncDreamer/resolve/main/ViT-L-14.pt"
}

for fname, url in CHECKPOINTS.items():
    fpath = CKPT_DIR / fname
    if not fpath.exists():
        print(f"📥 Downloading {fname}...")
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d "{CKPT_DIR}" -o "{fname}"
    else:
        print(f"✅ {fname} exists")

sys.path.insert(0, str(SYNCDREAMER_PATH))
os.chdir(SYNCDREAMER_PATH)

In [ ]:
import gc
import torch
from pathlib import Path
from omegaconf import OmegaConf
from ldm.util import instantiate_from_config

# ⚠️ CRITICAL: Aggressive memory cleanup before loading SyncDreamer
# T4 has limited RAM (~12GB on Colab free tier)
print("🧹 Clearing memory before loading SyncDreamer...")
gc.collect()
gc.collect()
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

# Check available memory
import psutil
ram_available = psutil.virtual_memory().available / 1024**3
print(f"   Available RAM: {ram_available:.1f} GB")
if ram_available < 5:
    print("   ⚠️ Low RAM warning! Consider restarting runtime.")

# Load SyncDreamer model
print("\n📥 Loading SyncDreamer model...")

config_path = SYNCDREAMER_PATH / "configs" / "syncdreamer.yaml"
config = OmegaConf.load(config_path)

# Instantiate model from config
syncdreamer_model = instantiate_from_config(config.model)

# Load pretrained weights with MEMORY-MAPPED approach
ckpt_path = CKPT_DIR / "syncdreamer-pretrain.ckpt"
print(f"   Loading checkpoint: {ckpt_path}")

# ⚠️ MEMORY FIX: Use mmap=True to avoid loading entire file into RAM
# This memory-maps the file instead of loading it all at once
try:
    # Try mmap first (PyTorch 2.1+)
    print("   Using memory-mapped loading (mmap=True)...")
    checkpoint = torch.load(ckpt_path, map_location="cpu", mmap=True, weights_only=False)
except TypeError:
    # Fallback for older PyTorch without mmap support
    print("   Fallback: Standard loading...")
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)

# Extract state dict
if "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
else:
    state_dict = checkpoint

# Load weights
print("   Loading state dict into model...")
missing, unexpected = syncdreamer_model.load_state_dict(state_dict, strict=False)

# ⚠️ CRITICAL: Delete checkpoint from RAM immediately
print("   Cleaning up checkpoint from RAM...")
del checkpoint
del state_dict
gc.collect()
gc.collect()

if missing:
    print(f"   ⚠️ Missing keys: {len(missing)}")
if unexpected:
    print(f"   ⚠️ Unexpected keys: {len(unexpected)}")

# Check RAM after loading
ram_after = psutil.virtual_memory().available / 1024**3
print(f"   RAM after loading: {ram_after:.1f} GB")

# Move to GPU
print("   Moving model to GPU...")
syncdreamer_model = syncdreamer_model.cuda().eval()

# Final cleanup
gc.collect()
torch.cuda.empty_cache()

# Verify model is ready
print(f"\n✅ SyncDreamer loaded!")
print(f"   Model type: {type(syncdreamer_model).__name__}")
print(f"   GPU memory: {torch.cuda.memory_allocated()/1024**3:.1f} GB allocated")

In [ ]:
from ldm.models.diffusion.sync_dreamer import SyncDDIMSampler
from ldm.util import prepare_inputs  # CRITICAL: Use official data preparation

# ✅ FIXED: Camera configuration MUST match SyncDreamer training data
# SyncDreamer generates 16 views at FIXED 30° elevation, azimuths spaced 22.5° apart
ELEVATIONS = [30.0] * 16  # All 16 views at 30° elevation
AZIMUTHS = [i * 22.5 for i in range(16)]  # 0°, 22.5°, 45°, ..., 337.5°
RADIUS = 1.5

# ✅ FIXED: Use official prepare_inputs function for proper data preparation
# This handles alpha channel, CLIP embedding, and proper normalization
processed_path = DIRS['triposr'] / "processed_input.png"

INPUT_ELEVATION = 30.0  # Assume front view at 30 degrees
CROP_SIZE = 200         # Crop foreground to this size

print(f"📸 Preparing input: {processed_path}")
print(f"   Input elevation: {INPUT_ELEVATION}°")
print(f"   Crop size: {CROP_SIZE}")

# Verify image has alpha channel (required by prepare_inputs)
img_check = Image.open(str(processed_path))
print(f"   Image format: {img_check.mode} (channels: {len(img_check.getbands())})")
if img_check.mode != 'RGBA':
    print("   ⚠️ Converting to RGBA...")
    img_check = img_check.convert('RGBA')
    img_check.save(str(processed_path))
img_check.close()

# Use official SyncDreamer data preparation
data = prepare_inputs(str(processed_path), INPUT_ELEVATION, CROP_SIZE)

# Move to GPU and add batch dimension
for k, v in data.items():
    data[k] = v.unsqueeze(0).cuda()
    print(f"   {k}: {data[k].shape}")

print(f"\n✅ Input prepared using official prepare_inputs()")

In [ ]:
# Run SyncDreamer inference
print("\n🚀 Running SyncDreamer (this takes ~2-3 minutes)...")
start_time = time.time()

# Settings
SAMPLE_STEPS = 50
CFG_SCALE = 2.0

sampler = SyncDDIMSampler(syncdreamer_model, SAMPLE_STEPS)

try:
    with torch.no_grad():
        # ✅ FIXED: Data already prepared correctly by prepare_inputs()
        # Run synchronized multi-view generation
        x_sample = syncdreamer_model.sample(
            sampler, 
            data, 
            CFG_SCALE, 
            BATCH_VIEW_NUM
        )
        # x_sample shape: (B, N, C, H, W) where N=16 views
        
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        print("⚠️ OOM Error! Reducing batch size and retrying...")
        clear_gpu()
        BATCH_VIEW_NUM = max(1, BATCH_VIEW_NUM // 2)
        print(f"   New BATCH_VIEW_NUM: {BATCH_VIEW_NUM}")
        
        sampler = SyncDDIMSampler(syncdreamer_model, SAMPLE_STEPS)
        with torch.no_grad():
            x_sample = syncdreamer_model.sample(sampler, data, CFG_SCALE, BATCH_VIEW_NUM)
    else:
        raise

elapsed = time.time() - start_time
print(f"\n✅ SyncDreamer completed in {elapsed/60:.1f} minutes")
print(f"   Output shape: {x_sample.shape}")

In [ ]:
# Save multi-view images
print("\n💾 Saving multi-view images...")

# Convert samples to images: [-1,1] -> [0,1]
samples = (x_sample.clamp(-1, 1) + 1) / 2

syncdreamer_views = []

B, N, C, H, W = samples.shape
print(f"   Processing {N} views at {H}x{W}")

for i in range(N):
    img_tensor = samples[0, i]  # (C, H, W)
    img_np = (img_tensor.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    img_pil = Image.fromarray(img_np)
    
    # Save individual view
    elev = int(ELEVATIONS[i])
    azim = int(AZIMUTHS[i])
    save_path = DIRS['syncdreamer'] / f"view_{i:02d}_e{elev}_a{azim}.png"
    img_pil.save(save_path)
    syncdreamer_views.append(img_pil)

print(f"✅ Saved {len(syncdreamer_views)} views to {DIRS['syncdreamer']}")

# Display grid
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for i, ax in enumerate(axes.flat):
    ax.imshow(syncdreamer_views[i])
    ax.set_title(f"E={int(ELEVATIONS[i])}° A={int(AZIMUTHS[i])}°", fontsize=8)
    ax.axis('off')
plt.suptitle("SyncDreamer: 16 Multi-View Images", fontsize=14)
plt.tight_layout()
plt.savefig(DIRS['syncdreamer'] / "grid.png", dpi=150)
plt.show()

# Cleanup SyncDreamer to free VRAM
del syncdreamer_model, sampler, x_sample, samples
clear_gpu()

# Stage 4: SDXL Enhancement (Optional)

**Input:** Multi-view images  
**Output:** Enhanced multi-view images  
**Time:** ~1 minute per image

Skip this stage if you want faster results.

In [ ]:
SKIP_ENHANCEMENT = False  # Set to True to skip this stage

# ⚠️ CRITICAL: Check RAM BEFORE attempting to load SDXL
# SDXL UNet alone requires ~5-6GB RAM to load, plus existing Python overhead
import gc
import psutil

# Aggressive cleanup first
for _ in range(10):
    gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

ram_available = psutil.virtual_memory().available / 1024**3
ram_total = psutil.virtual_memory().total / 1024**3

print(f"\n📊 Memory Status:")
print(f"   Available RAM: {ram_available:.1f} GB / {ram_total:.1f} GB")
print(f"   SDXL UNet requires: ~6 GB RAM to load")

# Auto-skip if RAM is insufficient (need ~8GB free to be safe)
if ram_available < 7:
    print(f"\n⚠️ AUTO-SKIP: Insufficient RAM for SDXL ({ram_available:.1f} GB < 7 GB)")
    print("   T4 GPUs on Colab free tier often have limited RAM.")
    print("   Skipping SDXL enhancement to prevent crash.")
    SKIP_ENHANCEMENT = True

if not SKIP_ENHANCEMENT:
    print("\n" + "="*60)
    print("✨ STAGE 4: SDXL Lightning Enhancement")
    print("="*60)
    
    # Save view list to disk temporarily, then delete from RAM
    import pickle
    views_cache_path = DIRS['syncdreamer'] / "_views_cache.pkl"
    with open(views_cache_path, 'wb') as f:
        pickle.dump(syncdreamer_views, f)
    del syncdreamer_views
    
    # Delete any lingering objects
    for var_name in ['data', 'x_sample', 'samples', 'sampler']:
        try:
            exec(f'del {var_name}')
        except:
            pass
    
    # Aggressive garbage collection
    for _ in range(10):
        gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    
    from diffusers import StableDiffusionXLImg2ImgPipeline, AutoencoderKL, EulerDiscreteScheduler
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file
    
    print("\n📥 Loading SDXL Lightning...")
    print("   Using single-file LoRA method (lower RAM usage)")
    
    base_model = "stabilityai/stable-diffusion-xl-base-1.0"
    repo = "ByteDance/SDXL-Lightning"
    
    try:
        # ✅ MEMORY-OPTIMIZED: Load pipeline with low_cpu_mem_usage
        print("   Loading base pipeline (this may take a minute)...")
        pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
            base_model,
            torch_dtype=torch.float16,
            variant="fp16",
            use_safetensors=True,
            low_cpu_mem_usage=True,  # ⚠️ CRITICAL: Reduces RAM during loading
        )
        
        # Apply fp16-fixed VAE
        print("   Loading fp16-fixed VAE...")
        pipe.vae = AutoencoderKL.from_pretrained(
            "madebyollin/sdxl-vae-fp16-fix",
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
        )
        
        # Download and load Lightning LoRA (much smaller than full UNet swap)
        print("   Applying Lightning LoRA weights...")
        lora_path = hf_hub_download(repo, "sdxl_lightning_4step_lora.safetensors")
        pipe.load_lora_weights(lora_path)
        pipe.fuse_lora()  # Fuse for faster inference
        
        # Use correct scheduler for Lightning
        pipe.scheduler = EulerDiscreteScheduler.from_config(
            pipe.scheduler.config, 
            timestep_spacing="trailing"
        )
        
        # Move to GPU
        pipe = pipe.to("cuda")
        
        # Cleanup
        gc.collect()
        torch.cuda.empty_cache()
        
        print("✅ SDXL Lightning loaded (LoRA method)!")
        print("   ⚠️ Remember: guidance_scale MUST be 0 for Lightning")
        
        # Reload syncdreamer_views from cache
        print("   Loading cached views...")
        with open(views_cache_path, 'rb') as f:
            syncdreamer_views = pickle.load(f)
        views_cache_path.unlink()  # Delete cache file
        
    except Exception as e:
        print(f"❌ Failed to load SDXL Lightning: {e}")
        print("   Falling back to skip enhancement")
        SKIP_ENHANCEMENT = True
        
        # Reload syncdreamer_views if enhancement failed
        try:
            with open(views_cache_path, 'rb') as f:
                syncdreamer_views = pickle.load(f)
            views_cache_path.unlink()
        except:
            # Reload from disk as fallback
            syncdreamer_views = []
            for i in range(16):
                img_path = DIRS['syncdreamer'] / f"view_{i:02d}_e30_a{int(i*22.5)}.png"
                syncdreamer_views.append(Image.open(img_path))
else:
    print("\n⏭️ Skipping SDXL enhancement stage")
    print("   Using original SyncDreamer views (still high quality!)")
    
    # Make sure syncdreamer_views is available for next stage
    if 'syncdreamer_views' not in dir() or syncdreamer_views is None:
        print("   Reloading views from disk...")
        syncdreamer_views = []
        for i in range(16):
            img_path = DIRS['syncdreamer'] / f"view_{i:02d}_e30_a{int(i*22.5)}.png"
            syncdreamer_views.append(Image.open(img_path))

In [ ]:
if not SKIP_ENHANCEMENT:
    # Enhance select views (not all 16 to save time)
    VIEWS_TO_ENHANCE = [0, 4, 8, 12]  # Only 4 views to reduce memory pressure
    
    print(f"\n🚀 Enhancing {len(VIEWS_TO_ENHANCE)} views...")
    
    enhanced_views = syncdreamer_views.copy()  # Start with original
    
    prompt = "highly detailed 3D render, professional studio lighting, sharp textures, photorealistic, 8k quality"
    negative_prompt = "blurry, low quality, artifacts, noise, watermark, text"
    
    for i, view_idx in enumerate(VIEWS_TO_ENHANCE):
        print(f"  Enhancing view {view_idx} ({i+1}/{len(VIEWS_TO_ENHANCE)})...")
        
        # Resize input for SDXL (works best at 512-1024)
        input_img = syncdreamer_views[view_idx].resize((512, 512), Image.LANCZOS)
        
        with torch.no_grad():
            result = pipe(
                prompt=prompt,
                negative_prompt=negative_prompt,
                image=input_img,
                strength=0.35,  # Lower = preserve more original structure
                num_inference_steps=4,  # Lightning uses 4 steps
                guidance_scale=0,  # Lightning uses CFG=0
            ).images[0]
        
        # Resize back to match SyncDreamer output size
        result_resized = result.resize((256, 256), Image.LANCZOS)
        enhanced_views[view_idx] = result_resized
        result.save(DIRS['enhanced'] / f"enhanced_{view_idx:02d}.png")
        
        # Clear VRAM between images
        torch.cuda.empty_cache()
    
    print(f"\n✅ Enhanced views saved to {DIRS['enhanced']}")
    
    # Cleanup SDXL to free VRAM for gsplat
    del pipe
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print("🧹 SDXL cleaned up")
else:
    # Use original SyncDreamer views (already high quality)
    enhanced_views = syncdreamer_views
    print("✅ Using original SyncDreamer views (no enhancement)")
    print("   Note: SyncDreamer views are already high quality for gsplat optimization")

# Stage 5A: Depth Estimation with MiDaS/DPT

**Input:** Multi-view images  
**Output:** Depth maps for each view  
**Time:** ~30 seconds per view

Depth estimation is **critical** for:
1. Better Gaussian initialization
2. Depth-consistent back-projection in MVCRM
3. Camera-space reasoning for refinement

We use Intel's DPT-Large (MiDaS 3.0) for high-quality monocular depth estimation.

In [ ]:
print("\n" + "="*60)
print("🎯 STAGE 5A: MiDaS/DPT Depth Estimation")
print("="*60)

import torch
import numpy as np
from PIL import Image
import torch.nn.functional as F
import gc

# ============================================================
# Depth Estimation Class - Supports both torch.hub and transformers
# ============================================================

class DepthEstimator:
    """
    Robust depth estimation using MiDaS/DPT models.
    
    Supports two backends:
    1. torch.hub (original MiDaS)
    2. transformers (Hugging Face DPT)
    
    The transformers backend is preferred for better API stability.
    """
    
    def __init__(self, backend="transformers", model_type="DPT_Large", device=None):
        """
        Initialize depth estimator.
        
        Args:
            backend: "transformers" (recommended) or "torch_hub"
            model_type: Model variant:
                - transformers: "Intel/dpt-large", "Intel/dpt-hybrid-midas"
                - torch_hub: "DPT_Large", "DPT_Hybrid", "MiDaS_small"
            device: "cuda" or "cpu" (auto-detect if None)
        """
        self.backend = backend
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = None
        self.processor = None
        self.transform = None
        
        if backend == "transformers":
            self._init_transformers(model_type)
        else:
            self._init_torch_hub(model_type)
    
    def _init_transformers(self, model_name):
        """Initialize using Hugging Face transformers."""
        from transformers import DPTImageProcessor, DPTForDepthEstimation
        
        # Map simple names to HuggingFace model IDs
        model_map = {
            "DPT_Large": "Intel/dpt-large",
            "DPT_Hybrid": "Intel/dpt-hybrid-midas", 
            "Intel/dpt-large": "Intel/dpt-large",
            "Intel/dpt-hybrid-midas": "Intel/dpt-hybrid-midas",
        }
        model_id = model_map.get(model_name, model_name)
        
        print(f"📥 Loading DPT model from Hugging Face: {model_id}")
        
        self.processor = DPTImageProcessor.from_pretrained(model_id)
        self.model = DPTForDepthEstimation.from_pretrained(model_id)
        self.model.to(self.device)
        self.model.eval()
        
        # Use half precision on GPU for faster inference
        if self.device == "cuda":
            self.model = self.model.half()
        
        print(f"✅ DPT model loaded on {self.device}")
    
    def _init_torch_hub(self, model_type):
        """Initialize using torch.hub (original MiDaS)."""
        print(f"📥 Loading MiDaS via torch.hub: {model_type}")
        
        self.model = torch.hub.load("intel-isl/MiDaS", model_type)
        self.model.to(self.device)
        self.model.eval()
        
        # Load appropriate transform
        midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
        if model_type in ["DPT_Large", "DPT_Hybrid"]:
            self.transform = midas_transforms.dpt_transform
        else:
            self.transform = midas_transforms.small_transform
        
        print(f"✅ MiDaS model loaded on {self.device}")
    
    def estimate(self, image, normalize=True):
        """
        Estimate depth from an image.
        
        Args:
            image: PIL Image, numpy array (H, W, 3), or path string
            normalize: If True, normalize output to [0, 1] where 1 = closest
        
        Returns:
            depth_map: numpy array (H, W), float32
        """
        # Convert input to PIL Image
        if isinstance(image, str):
            image = Image.open(image).convert("RGB")
        elif isinstance(image, np.ndarray):
            image = Image.fromarray(image.astype(np.uint8)).convert("RGB")
        elif not isinstance(image, Image.Image):
            raise TypeError(f"Unsupported image type: {type(image)}")
        
        original_size = image.size  # (W, H)
        
        if self.backend == "transformers":
            return self._estimate_transformers(image, original_size, normalize)
        else:
            return self._estimate_torch_hub(image, original_size, normalize)
    
    def _estimate_transformers(self, image, original_size, normalize):
        """Estimate using transformers backend."""
        # Preprocess
        inputs = self.processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(self.device)
        
        if self.device == "cuda":
            pixel_values = pixel_values.half()
        
        # Inference
        with torch.no_grad():
            outputs = self.model(pixel_values)
            predicted_depth = outputs.predicted_depth
        
        # Interpolate to original size
        # Note: original_size is (W, H), but interpolate expects (H, W)
        prediction = F.interpolate(
            predicted_depth.unsqueeze(1).float(),
            size=(original_size[1], original_size[0]),  # (H, W)
            mode="bicubic",
            align_corners=False,
        ).squeeze()
        
        depth = prediction.cpu().numpy()
        
        if normalize:
            depth = self._normalize_depth(depth)
        
        return depth.astype(np.float32)
    
    def _estimate_torch_hub(self, image, original_size, normalize):
        """Estimate using torch.hub backend."""
        # Convert to numpy for transform
        img_np = np.array(image)
        
        # Apply MiDaS transform
        input_tensor = self.transform(img_np).to(self.device)
        
        # Inference
        with torch.no_grad():
            prediction = self.model(input_tensor)
            
            # Interpolate to original size
            prediction = F.interpolate(
                prediction.unsqueeze(1),
                size=(original_size[1], original_size[0]),  # (H, W)
                mode="bicubic",
                align_corners=False,
            ).squeeze()
        
        depth = prediction.cpu().numpy()
        
        if normalize:
            depth = self._normalize_depth(depth)
        
        return depth.astype(np.float32)
    
    def _normalize_depth(self, depth):
        """
        Normalize depth to [0, 1] range.
        
        MiDaS outputs inverse depth (higher = closer).
        After normalization: 1 = closest, 0 = farthest
        """
        d_min, d_max = depth.min(), depth.max()
        if d_max - d_min > 1e-8:
            return (depth - d_min) / (d_max - d_min)
        return np.zeros_like(depth)
    
    def estimate_batch(self, images, normalize=True):
        """Estimate depth for multiple images."""
        return [self.estimate(img, normalize=normalize) for img in images]


# ============================================================
# Initialize Depth Estimator
# ============================================================

# Try transformers backend first, fallback to torch.hub
try:
    print("\n📦 Attempting to load DPT via Hugging Face transformers...")
    depth_estimator = DepthEstimator(
        backend="transformers",
        model_type="Intel/dpt-large",
        device="cuda"
    )
except Exception as e:
    print(f"⚠️ Transformers backend failed: {e}")
    print("📦 Falling back to torch.hub...")
    depth_estimator = DepthEstimator(
        backend="torch_hub",
        model_type="DPT_Large",
        device="cuda"
    )

# ============================================================
# Generate Depth Maps for All Views
# ============================================================

print("\n🔍 Generating depth maps for all views...")
depth_maps = []

# Use enhanced_views if available, otherwise syncdreamer_views
views_to_process = enhanced_views if 'enhanced_views' in dir() else syncdreamer_views

for i, view in enumerate(tqdm(views_to_process, desc="Estimating depth")):
    depth = depth_estimator.estimate(view, normalize=True)
    depth_maps.append(depth)
    
    # Save depth visualization for first few views
    if i < 4:
        import matplotlib.pyplot as plt
        depth_vis = (depth * 255).astype(np.uint8)
        depth_pil = Image.fromarray(depth_vis)
        depth_pil.save(DIRS['gsplat'] / f"depth_view_{i:02d}.png")

print(f"✅ Generated {len(depth_maps)} depth maps")
print(f"   Shape: {depth_maps[0].shape}")
print(f"   Range: [{depth_maps[0].min():.4f}, {depth_maps[0].max():.4f}]")

# Visualize sample depth maps
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(min(8, len(depth_maps))):
    row, col = i // 4, i % 4
    axes[row, col].imshow(depth_maps[i], cmap='magma')
    axes[row, col].set_title(f"View {i} Depth")
    axes[row, col].axis('off')
plt.suptitle("MiDaS Depth Estimation Results", fontsize=14)
plt.tight_layout()
plt.savefig(DIRS['gsplat'] / "depth_grid.png", dpi=150)
plt.show()

# Cleanup to free VRAM
del depth_estimator
gc.collect()
torch.cuda.empty_cache()
print("🧹 Depth estimator cleaned up")

# Stage 5: gsplat Optimization

**Input:** Initial Gaussian PLY + Multi-view images  
**Output:** Optimized Gaussian Splats  
**Time:** ~5 minutes

In [ ]:
print("\n" + "="*60)
print("🔮 STAGE 5: gsplat Optimization")
print("="*60)

import torch.nn as nn
from gsplat import rasterization
import math

device = torch.device("cuda:0")

# ⚠️ PRE-COMPILE GSPLAT CUDA KERNELS
# gsplat uses JIT compilation - first call takes 5-10 minutes on T4
# We do a dummy render here so the compilation happens with a clear progress message
print("\n⏳ Pre-compiling gsplat CUDA kernels...")
print("   This takes 5-10 minutes on first run (one-time per session)")
print("   You'll see 'Setting up CUDA...' - this is normal!")

# Dummy tensors for compilation trigger
_dummy_means = torch.zeros(100, 3, device=device)
_dummy_quats = torch.tensor([[1, 0, 0, 0]] * 100, dtype=torch.float32, device=device)
_dummy_scales = torch.ones(100, 3, device=device) * 0.01
_dummy_opacities = torch.ones(100, device=device)
_dummy_colors = torch.ones(100, 3, device=device)
_dummy_viewmat = torch.eye(4, device=device).unsqueeze(0)
_dummy_K = torch.tensor([[128, 0, 64], [0, 128, 64], [0, 0, 1]], dtype=torch.float32, device=device).unsqueeze(0)

try:
    _ = rasterization(
        means=_dummy_means,
        quats=_dummy_quats,
        scales=_dummy_scales,
        opacities=_dummy_opacities,
        colors=_dummy_colors,
        viewmats=_dummy_viewmat,
        Ks=_dummy_K,
        width=128,
        height=128,
        packed=False,
        render_mode="RGB",
    )
    print("✅ gsplat CUDA kernels compiled!")
except Exception as e:
    print(f"⚠️ Pre-compilation note: {e}")

# Cleanup dummy tensors
del _dummy_means, _dummy_quats, _dummy_scales, _dummy_opacities, _dummy_colors, _dummy_viewmat, _dummy_K
torch.cuda.empty_cache()

# Image size for rendering (matches SyncDreamer output)
IMAGE_SIZE = 256

# Load initial Gaussians from PLY
def load_gaussian_ply(path):
    """Load Gaussian parameters from PLY file."""
    plydata = PlyData.read(path)
    vertex = plydata['vertex']
    
    xyz = np.stack([vertex['x'], vertex['y'], vertex['z']], axis=-1)
    f_dc = np.stack([vertex['f_dc_0'], vertex['f_dc_1'], vertex['f_dc_2']], axis=-1)
    
    # Load f_rest if present
    f_rest_names = [f'f_rest_{i}' for i in range(45)]
    available_f_rest = [name for name in f_rest_names if name in vertex.data.dtype.names]
    if available_f_rest:
        f_rest = np.stack([vertex[name] for name in available_f_rest], axis=-1)
    else:
        f_rest = np.zeros((len(xyz), 45), dtype=np.float32)
    
    # ✅ FIXED: Ensure opacity is 1D (N,) - gsplat requires this shape
    opacity = np.asarray(vertex['opacity'], dtype=np.float32).flatten()
    scales = np.stack([vertex['scale_0'], vertex['scale_1'], vertex['scale_2']], axis=-1)
    rotations = np.stack([vertex['rot_0'], vertex['rot_1'], vertex['rot_2'], vertex['rot_3']], axis=-1)
    
    return {
        'xyz': torch.tensor(xyz, dtype=torch.float32),
        'f_dc': torch.tensor(f_dc, dtype=torch.float32),
        'f_rest': torch.tensor(f_rest, dtype=torch.float32),
        'opacity': torch.tensor(opacity, dtype=torch.float32),  # Shape: (N,)
        'scales': torch.tensor(scales, dtype=torch.float32),
        'rotations': torch.tensor(rotations, dtype=torch.float32),
    }

gaussians = load_gaussian_ply(str(INITIAL_PLY_PATH))
print(f"\n✅ Loaded {len(gaussians['xyz']):,} Gaussians")

In [ ]:
class GaussianModel(nn.Module):
    """
    PyTorch module for optimizable 3D Gaussian Splat parameters.
    
    ⚠️ CRITICAL: gsplat v1.0+ uses the following conventions:
    - Quaternions: wxyz format (w, x, y, z) where w is the scalar part
    - Opacity: 1D tensor of shape (N,) NOT (N, 1)
    - Scales: exp(scale_raw) for positive scales
    - Colors: 0.5 + C0 * f_dc for SH DC coefficient decoding
    """
    
    def __init__(self, gaussians):
        super().__init__()
        
        # Position parameters
        self.xyz = nn.Parameter(gaussians['xyz'].clone())
        
        # Color parameters (SH DC coefficients)
        self.f_dc = nn.Parameter(gaussians['f_dc'].clone())
        self.f_rest = nn.Parameter(gaussians['f_rest'].clone())
        
        # ✅ FIXED: Ensure opacity is 1D (N,) - gsplat requires this shape
        opacity_tensor = gaussians['opacity'].clone()
        if opacity_tensor.dim() > 1:
            opacity_tensor = opacity_tensor.squeeze(-1)
        self.opacity_raw = nn.Parameter(opacity_tensor)
        
        # Scale parameters (log space)
        self.scales_raw = nn.Parameter(gaussians['scales'].clone())
        
        # Rotation parameters (quaternions in wxyz format)
        self.rotations = nn.Parameter(gaussians['rotations'].clone())
        
        # Store initial bounds for regularization
        self._register_bounds()
    
    def _register_bounds(self):
        """Register min/max bounds for regularization."""
        with torch.no_grad():
            self.register_buffer('xyz_min', self.xyz.min(dim=0).values)
            self.register_buffer('xyz_max', self.xyz.max(dim=0).values)
    
    @property
    def opacity(self):
        """
        Get opacity values in [0, 1] range.
        ✅ Returns (N,) shape - required by gsplat.rasterization()
        """
        return torch.sigmoid(self.opacity_raw)
    
    @property
    def scales(self):
        """
        Get scale values (always positive via exp).
        ✅ Clamp to prevent numerical issues.
        """
        # Clamp raw values to prevent extreme scales
        clamped = torch.clamp(self.scales_raw, min=-10.0, max=5.0)
        return torch.exp(clamped)
    
    @property
    def quats_normalized(self):
        """
        Get normalized quaternions (required by gsplat).
        ✅ Normalizes to unit quaternions.
        """
        return self.rotations / (torch.norm(self.rotations, dim=-1, keepdim=True) + 1e-8)
    
    def get_colors(self):
        """
        Decode SH DC coefficients to RGB colors.
        Formula: color = 0.5 + C0 * f_dc
        """
        C0 = 0.28209479177387814  # First spherical harmonic coefficient
        colors = 0.5 + C0 * self.f_dc
        # Clamp to valid RGB range
        return torch.clamp(colors, 0.0, 1.0)
    
    def forward(self):
        """
        Return all Gaussian parameters in a dictionary.
        Used for rendering.
        """
        return {
            'xyz': self.xyz,
            'colors': self.get_colors(),
            'opacity': self.opacity,  # (N,) shape
            'scales': self.scales,    # (N, 3) shape
            'rotations': self.quats_normalized,  # (N, 4) normalized wxyz
        }
    
    def regularization_loss(self, lambda_scale=0.01, lambda_opacity=0.001):
        """
        Compute regularization losses to prevent degenerate solutions.
        
        - Scale regularization: prevent scales from becoming too large or too small
        - Opacity regularization: encourage binary opacity (fully visible or invisible)
        """
        # Scale regularization - penalize very large or very small scales
        scales = self.scales
        scale_loss = ((scales - 0.01).clamp(min=0) ** 2).mean()  # Penalize scales > 0.01
        scale_loss += ((0.0001 - scales).clamp(min=0) ** 2).mean()  # Penalize scales < 0.0001
        
        # Opacity regularization - encourage values near 0 or 1
        opacity = self.opacity
        opacity_loss = (opacity * (1 - opacity)).mean()  # Binary cross-entropy-like
        
        return lambda_scale * scale_loss + lambda_opacity * opacity_loss
    
    def get_param_stats(self):
        """Return statistics about current parameters (for debugging)."""
        with torch.no_grad():
            return {
                'xyz_mean': self.xyz.mean().item(),
                'xyz_std': self.xyz.std().item(),
                'opacity_mean': self.opacity.mean().item(),
                'opacity_min': self.opacity.min().item(),
                'opacity_max': self.opacity.max().item(),
                'scales_mean': self.scales.mean().item(),
                'scales_min': self.scales.min().item(),
                'scales_max': self.scales.max().item(),
                'colors_mean': self.get_colors().mean().item(),
            }


# Initialize model with loaded Gaussians
model = GaussianModel(gaussians).to(device)

# Print model summary
print(f"✅ GaussianModel initialized:")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Gaussians: {len(model.xyz):,}")

# Print initial parameter statistics
stats = model.get_param_stats()
print(f"\n📊 Initial Parameter Stats:")
print(f"   XYZ mean: {stats['xyz_mean']:.4f}, std: {stats['xyz_std']:.4f}")
print(f"   Opacity: [{stats['opacity_min']:.4f}, {stats['opacity_max']:.4f}], mean: {stats['opacity_mean']:.4f}")
print(f"   Scales: [{stats['scales_min']:.6f}, {stats['scales_max']:.6f}], mean: {stats['scales_mean']:.6f}")
print(f"   Colors mean: {stats['colors_mean']:.4f}")

In [ ]:
# ============================================================
# Camera System - Matching SyncDreamer Conventions
# ============================================================
# SyncDreamer uses: Y-up coordinate system, camera looks at origin

# ⚠️ CRITICAL: First check where the Gaussians actually are
with torch.no_grad():
    xyz = model.xyz.cpu().numpy()
    gaussian_center = xyz.mean(axis=0)
    gaussian_extent = (xyz.max(axis=0) - xyz.min(axis=0)).max()
    gaussian_radius = np.linalg.norm(xyz - gaussian_center, axis=1).max()
    
print(f"📊 Gaussian Point Cloud Analysis:")
print(f"   Points: {len(xyz):,}")
print(f"   Center: ({gaussian_center[0]:.3f}, {gaussian_center[1]:.3f}, {gaussian_center[2]:.3f})")
print(f"   Extent: {gaussian_extent:.3f}")
print(f"   Max radius from center: {gaussian_radius:.3f}")

# ⚠️ FIX: Recenter Gaussians to origin if needed
if np.linalg.norm(gaussian_center) > 0.05:
    print(f"\n🔧 Recentering Gaussians to origin...")
    with torch.no_grad():
        offset = torch.tensor(gaussian_center, device=device, dtype=torch.float32)
        model.xyz.data -= offset
    gaussian_center = np.array([0.0, 0.0, 0.0])
    print(f"   New center: (0, 0, 0)")

# ⚠️ FIX: Normalize scale to unit sphere if too large or too small
if gaussian_extent > 2.0 or gaussian_extent < 0.1:
    scale_factor = 1.0 / max(gaussian_extent, 0.01)
    print(f"\n🔧 Normalizing Gaussian cloud scale...")
    print(f"   Scale factor: {scale_factor:.4f}")
    with torch.no_grad():
        model.xyz.data *= scale_factor
        # Also scale the Gaussian sizes
        model.scales_raw.data += np.log(scale_factor)
    gaussian_extent = 1.0
    print(f"   New extent: ~1.0")

# Calculate optimal camera radius based on model size
# Camera should be ~2.5x the model extent for good framing
RADIUS = max(gaussian_extent * 2.5, 1.5)
print(f"\n📸 Camera Setup:")
print(f"   Optimal radius: {RADIUS:.3f}")


def create_camera_pose(elevation_deg, azimuth_deg, radius=1.5, look_at=None):
    """
    Create world-to-camera (w2c) matrix for given elevation and azimuth.
    
    Coordinate system: Y-up, camera looks at look_at point
    
    Args:
        elevation_deg: Elevation angle in degrees (0 = horizon, 90 = top)
        azimuth_deg: Azimuth angle in degrees (0 = front, 90 = right)
        radius: Distance from look_at point
        look_at: Point the camera looks at (default: origin)
    
    Returns:
        w2c: (4, 4) world-to-camera transformation matrix
    """
    if look_at is None:
        look_at = np.array([0.0, 0.0, 0.0])
    
    elev = math.radians(elevation_deg)
    azim = math.radians(azimuth_deg)
    
    # Camera position in spherical coordinates (Y-up convention)
    # X = right, Y = up, Z = towards viewer (OpenGL convention)
    x = radius * math.cos(elev) * math.sin(azim)
    y = radius * math.sin(elev)
    z = radius * math.cos(elev) * math.cos(azim)
    
    cam_pos = np.array([x, y, z]) + look_at
    
    # Camera basis vectors
    up = np.array([0.0, 1.0, 0.0])  # Y-up
    
    # Forward direction (from camera to target)
    forward = look_at - cam_pos
    forward_norm = np.linalg.norm(forward)
    if forward_norm < 1e-6:
        forward = np.array([0.0, 0.0, -1.0])
    else:
        forward = forward / forward_norm
    
    # Right vector
    right = np.cross(forward, up)
    right_norm = np.linalg.norm(right)
    if right_norm < 1e-6:
        # Camera looking straight up/down - use alternative up vector
        up = np.array([0.0, 0.0, 1.0])
        right = np.cross(forward, up)
        right_norm = np.linalg.norm(right)
    right = right / (right_norm + 1e-8)
    
    # Recompute up vector to ensure orthogonality
    up_new = np.cross(right, forward)
    up_new = up_new / (np.linalg.norm(up_new) + 1e-8)
    
    # World-to-camera transformation
    # The camera coordinate system: X=right, Y=up, Z=-forward (OpenGL convention)
    R = np.eye(3, dtype=np.float32)
    R[0, :] = right
    R[1, :] = up_new
    R[2, :] = -forward  # Camera looks along -Z
    
    # Translation: t = -R @ cam_pos
    t = -R @ cam_pos
    
    # Construct 4x4 matrix
    w2c = np.eye(4, dtype=np.float32)
    w2c[:3, :3] = R
    w2c[:3, 3] = t
    
    return w2c


def get_intrinsics(fov_deg=49.1, image_size=256):
    """
    Get camera intrinsics matrix.
    
    Args:
        fov_deg: Field of view in degrees (vertical FOV)
        image_size: Image dimension (assumes square images)
    
    Returns:
        K: (3, 3) intrinsics matrix
    """
    fov_rad = math.radians(fov_deg)
    focal = image_size / (2 * math.tan(fov_rad / 2))
    
    K = np.array([
        [focal, 0, image_size / 2],
        [0, focal, image_size / 2],
        [0, 0, 1]
    ], dtype=np.float32)
    
    return K


# Pre-compute all camera poses (using SyncDreamer camera parameters)
# SyncDreamer uses 16 views at 30° elevation, spaced 22.5° in azimuth
camera_poses = []
for elev, azim in zip(ELEVATIONS, AZIMUTHS):
    pose = create_camera_pose(elev, azim, radius=RADIUS, look_at=gaussian_center)
    camera_poses.append(pose)

intrinsics = get_intrinsics(fov_deg=49.1, image_size=IMAGE_SIZE)

print(f"   Created {len(camera_poses)} camera poses")
print(f"   Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"   Focal length: {intrinsics[0,0]:.1f}")
print(f"   Principal point: ({intrinsics[0,2]:.0f}, {intrinsics[1,2]:.0f})")

# ============================================================
# ⚠️ DEBUG: Verify cameras can see the Gaussians
# ============================================================
print("\n🔍 Camera Visibility Check:")

# Define render function here to use for testing
def render_gaussians(model, w2c, K, image_size):
    """
    Render Gaussian splats from a camera viewpoint.
    
    Args:
        model: GaussianModel instance
        w2c: (4, 4) world-to-camera matrix
        K: (3, 3) intrinsics matrix
        image_size: Output image dimension
    
    Returns:
        render_colors: (H, W, 3) rendered RGB image
        render_alphas: (H, W, 1) alpha/opacity map
    """
    params = model()
    
    viewmat = torch.tensor(w2c, dtype=torch.float32, device=device)
    K_tensor = torch.tensor(K, dtype=torch.float32, device=device)
    
    try:
        render_colors, render_alphas, info = rasterization(
            means=params['xyz'],
            quats=params['rotations'],  # Already normalized
            scales=params['scales'],
            opacities=params['opacity'],  # (N,) shape
            colors=params['colors'],
            viewmats=viewmat.unsqueeze(0),
            Ks=K_tensor.unsqueeze(0),
            width=image_size,
            height=image_size,
            packed=False,
            render_mode="RGB",
        )
        return render_colors[0], render_alphas[0]
    except Exception as e:
        print(f"❌ Render error: {e}")
        # Return empty image on error
        return (
            torch.zeros(image_size, image_size, 3, device=device),
            torch.zeros(image_size, image_size, 1, device=device)
        )


# Test render from multiple views
with torch.no_grad():
    for test_idx in [0, 4, 8]:  # Front, side, back
        test_render, test_alpha = render_gaussians(model, camera_poses[test_idx], intrinsics, IMAGE_SIZE)
        mean_brightness = test_render.mean().item()
        max_brightness = test_render.max().item()
        alpha_coverage = (test_alpha > 0.5).float().mean().item()
        
        status = "✅" if mean_brightness > 0.05 else "⚠️"
        print(f"   View {test_idx}: brightness={mean_brightness:.4f}, max={max_brightness:.4f}, "
              f"alpha_coverage={alpha_coverage*100:.1f}% {status}")

# If all views are dark, there's a problem
with torch.no_grad():
    test_render, _ = render_gaussians(model, camera_poses[0], intrinsics, IMAGE_SIZE)
    if test_render.max().item() < 0.01:
        print("\n⚠️ WARNING: All views appear dark!")
        print("   Attempting automatic fixes...")
        
        # Fix 1: Increase scales
        print("   → Increasing Gaussian scales...")
        model.scales_raw.data += 2.0
        
        # Fix 2: Increase opacity
        print("   → Increasing opacity...")
        model.opacity_raw.data = torch.clamp(model.opacity_raw.data, min=2.0)
        
        # Fix 3: Ensure colors are visible
        print("   → Boosting color values...")
        with torch.no_grad():
            color_mean = model.get_colors().mean()
            if color_mean < 0.3:
                model.f_dc.data *= 1.5
        
        # Re-test
        test_render2, _ = render_gaussians(model, camera_poses[0], intrinsics, IMAGE_SIZE)
        print(f"\n   After fixes: brightness={test_render2.mean():.4f}")
    else:
        print("\n✅ Camera visibility verified - Gaussians are visible!")

In [ ]:
# ============================================================
# Render Function - Already defined above but adding debugging version
# ============================================================

def render_gaussians_debug(model, w2c, K, image_size, verbose=False):
    """
    Debug version of render function with extra diagnostics.
    
    Returns additional information about the rendering process.
    """
    params = model()
    
    viewmat = torch.tensor(w2c, dtype=torch.float32, device=device)
    K_tensor = torch.tensor(K, dtype=torch.float32, device=device)
    
    # Transform Gaussians to camera space for debugging
    means_homo = torch.cat([params['xyz'], torch.ones_like(params['xyz'][:, :1])], dim=1)
    means_cam = (viewmat @ means_homo.T).T[:, :3]
    
    # Check how many Gaussians are in front of camera
    in_front = (means_cam[:, 2] > 0.1).sum().item()
    total = len(means_cam)
    
    if verbose:
        print(f"   Gaussians in front of camera: {in_front}/{total} ({in_front/total*100:.1f}%)")
        print(f"   Camera Z range: [{means_cam[:, 2].min():.3f}, {means_cam[:, 2].max():.3f}]")
    
    try:
        render_colors, render_alphas, info = rasterization(
            means=params['xyz'],
            quats=params['rotations'],
            scales=params['scales'],
            opacities=params['opacity'],
            colors=params['colors'],
            viewmats=viewmat.unsqueeze(0),
            Ks=K_tensor.unsqueeze(0),
            width=image_size,
            height=image_size,
            packed=False,
            render_mode="RGB",
        )
        
        if verbose:
            print(f"   Render mean: {render_colors[0].mean():.4f}")
            print(f"   Render max: {render_colors[0].max():.4f}")
            print(f"   Alpha mean: {render_alphas[0].mean():.4f}")
        
        return render_colors[0], render_alphas[0], {
            'in_front': in_front,
            'total': total,
            'means_cam': means_cam
        }
    except Exception as e:
        print(f"❌ Render error: {e}")
        import traceback
        traceback.print_exc()
        return (
            torch.zeros(image_size, image_size, 3, device=device),
            torch.zeros(image_size, image_size, 1, device=device),
            {'error': str(e)}
        )


# Quick validation render
print("🔍 Validation Render:")
with torch.no_grad():
    rgb, alpha, debug_info = render_gaussians_debug(
        model, camera_poses[0], intrinsics, IMAGE_SIZE, verbose=True
    )
    
if rgb.max().item() > 0.01:
    print("✅ Render validation passed!")
else:
    print("⚠️ Render validation: Low brightness detected")

In [ ]:
# ============================================================
# 🔍 COMPREHENSIVE PRE-OPTIMIZATION VALIDATION
# ============================================================
# This cell validates all components before starting the expensive
# optimization loop. Catching issues here saves time!

print("\n" + "="*60)
print("🔍 PRE-OPTIMIZATION VALIDATION")
print("="*60)

validation_passed = True
issues = []

# ============================================================
# 1. Validate Gaussian Parameters
# ============================================================
print("\n1️⃣ Validating Gaussian Parameters...")

with torch.no_grad():
    params = model()
    
    # Check positions
    xyz = params['xyz']
    print(f"   XYZ shape: {xyz.shape}")
    print(f"   XYZ range: [{xyz.min():.4f}, {xyz.max():.4f}]")
    
    if torch.isnan(xyz).any():
        issues.append("❌ NaN in positions!")
        validation_passed = False
    
    # Check colors
    colors = params['colors']
    print(f"   Colors shape: {colors.shape}")
    print(f"   Colors range: [{colors.min():.4f}, {colors.max():.4f}]")
    
    if colors.max() < 0.1:
        issues.append("⚠️ Colors are very dark (max < 0.1)")
    
    # Check opacity
    opacity = params['opacity']
    print(f"   Opacity shape: {opacity.shape} (should be (N,))")
    print(f"   Opacity range: [{opacity.min():.4f}, {opacity.max():.4f}]")
    
    if opacity.dim() != 1:
        issues.append(f"❌ Opacity wrong shape: {opacity.shape}, expected (N,)")
        validation_passed = False
    
    if opacity.max() < 0.1:
        issues.append("⚠️ Opacity very low (max < 0.1) - Gaussians may be invisible")
    
    # Check scales
    scales = params['scales']
    print(f"   Scales shape: {scales.shape}")
    print(f"   Scales range: [{scales.min():.6f}, {scales.max():.6f}]")
    
    if scales.max() < 0.0001:
        issues.append("⚠️ Scales very small (max < 0.0001) - Gaussians may be invisible")
    
    # Check rotations
    rots = params['rotations']
    print(f"   Rotations shape: {rots.shape}")
    rot_norms = torch.norm(rots, dim=-1)
    print(f"   Rotation norms: [{rot_norms.min():.4f}, {rot_norms.max():.4f}] (should be ~1.0)")
    
    if (rot_norms < 0.9).any() or (rot_norms > 1.1).any():
        issues.append("⚠️ Some quaternions not normalized")

# ============================================================
# 2. Validate Camera Setup
# ============================================================
print("\n2️⃣ Validating Camera Setup...")

print(f"   Number of camera poses: {len(camera_poses)}")
print(f"   Number of target images: {len(target_tensors)}")

if len(camera_poses) != len(target_tensors):
    issues.append(f"❌ Camera count ({len(camera_poses)}) != target count ({len(target_tensors)})")
    validation_passed = False

# Check a camera matrix
sample_pose = camera_poses[0]
print(f"   Camera pose shape: {sample_pose.shape}")
print(f"   Camera translation: {sample_pose[:3, 3]}")

# Check intrinsics
print(f"   Intrinsics shape: {intrinsics.shape}")
print(f"   Focal length: {intrinsics[0, 0]:.1f}")

# ============================================================
# 3. Validate Target Images
# ============================================================
print("\n3️⃣ Validating Target Images...")

for i in range(min(3, len(target_tensors))):
    target = target_tensors[i]
    print(f"   Target {i}: shape={target.shape}, range=[{target.min():.3f}, {target.max():.3f}]")
    
    if target.shape != (IMAGE_SIZE, IMAGE_SIZE, 3):
        issues.append(f"❌ Target {i} wrong shape: {target.shape}")
        validation_passed = False

# ============================================================
# 4. Validate Depth Maps (if available)
# ============================================================
if depth_tensors:
    print("\n4️⃣ Validating Depth Maps...")
    print(f"   Number of depth maps: {len(depth_tensors)}")
    
    for i in range(min(3, len(depth_tensors))):
        depth = depth_tensors[i]
        print(f"   Depth {i}: shape={depth.shape}, range=[{depth.min():.3f}, {depth.max():.3f}]")
else:
    print("\n4️⃣ Depth Maps: Not available (depth guidance will be disabled)")

# ============================================================
# 5. Render Test
# ============================================================
print("\n5️⃣ Render Test...")

with torch.no_grad():
    test_renders = []
    for i in [0, 4, 8, 12]:
        rendered, alpha = render_gaussians(model, camera_poses[i], intrinsics, IMAGE_SIZE)
        mean_val = rendered.mean().item()
        max_val = rendered.max().item()
        test_renders.append((i, mean_val, max_val))
        status = "✅" if mean_val > 0.05 else "⚠️"
        print(f"   View {i}: mean={mean_val:.4f}, max={max_val:.4f} {status}")
        
        if max_val < 0.01:
            issues.append(f"⚠️ View {i} is completely black")

# ============================================================
# 6. Gradient Flow Test
# ============================================================
print("\n6️⃣ Gradient Flow Test...")

# Enable gradients temporarily
model.train()
rendered, alpha = render_gaussians(model, camera_poses[0], intrinsics, IMAGE_SIZE)
loss = F.mse_loss(rendered, target_tensors[0])
loss.backward()

# Check if gradients flow to all parameters
grad_params = []
for name, param in model.named_parameters():
    if param.grad is not None:
        grad_norm = param.grad.norm().item()
        grad_params.append((name, grad_norm))
        print(f"   {name}: grad_norm={grad_norm:.6f}")
    else:
        print(f"   {name}: NO GRADIENT ⚠️")
        issues.append(f"⚠️ No gradient for {name}")

model.eval()

# Clear gradients
for param in model.parameters():
    param.grad = None

# ============================================================
# Summary
# ============================================================
print("\n" + "="*60)
if validation_passed and len(issues) == 0:
    print("✅ ALL VALIDATIONS PASSED!")
    print("   Ready for optimization.")
else:
    print("⚠️ VALIDATION ISSUES DETECTED:")
    for issue in issues:
        print(f"   {issue}")
    
    if not validation_passed:
        print("\n❌ CRITICAL ISSUES - Optimization may fail!")
        print("   Please review and fix the issues above before continuing.")
    else:
        print("\n⚠️ Non-critical issues - Optimization may still work but quality may suffer.")
print("="*60)

In [ ]:
from tqdm import tqdm
import torch.nn.functional as F
import math

# ============================================================
# ⚠️ CRITICAL: Advanced Loss Functions for Better Optimization
# ============================================================

def ssim_loss(pred, target, window_size=11, C1=0.01**2, C2=0.03**2):
    """
    Compute Structural Similarity Index (SSIM) loss.
    Higher SSIM = more similar, so we return 1 - SSIM for loss.
    """
    # Ensure inputs are (B, C, H, W)
    if pred.dim() == 3:  # (H, W, C)
        pred = pred.permute(2, 0, 1).unsqueeze(0)  # (1, C, H, W)
        target = target.permute(2, 0, 1).unsqueeze(0)
    
    # Create gaussian window
    sigma = 1.5
    gauss = torch.tensor([
        math.exp(-(x - window_size//2)**2 / (2 * sigma**2))
        for x in range(window_size)
    ], dtype=pred.dtype, device=pred.device)
    gauss = gauss / gauss.sum()
    
    # 2D window
    window = gauss.unsqueeze(1) @ gauss.unsqueeze(0)
    window = window.unsqueeze(0).unsqueeze(0)  # (1, 1, k, k)
    window = window.expand(pred.shape[1], 1, window_size, window_size)
    
    # Compute means
    mu1 = F.conv2d(pred, window, padding=window_size//2, groups=pred.shape[1])
    mu2 = F.conv2d(target, window, padding=window_size//2, groups=target.shape[1])
    
    mu1_sq = mu1 ** 2
    mu2_sq = mu2 ** 2
    mu1_mu2 = mu1 * mu2
    
    # Compute variances and covariance
    sigma1_sq = F.conv2d(pred * pred, window, padding=window_size//2, groups=pred.shape[1]) - mu1_sq
    sigma2_sq = F.conv2d(target * target, window, padding=window_size//2, groups=target.shape[1]) - mu2_sq
    sigma12 = F.conv2d(pred * target, window, padding=window_size//2, groups=pred.shape[1]) - mu1_mu2
    
    # SSIM formula
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / \
               ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    
    return 1.0 - ssim_map.mean()


def combined_loss(pred, target, lambda_l1=0.8, lambda_ssim=0.2):
    """
    Combined L1 + SSIM loss for photometric reconstruction.
    L1 preserves gradients well, SSIM captures structural similarity.
    """
    l1 = F.l1_loss(pred, target)
    
    try:
        ssim = ssim_loss(pred, target)
    except Exception:
        # Fallback if SSIM fails
        ssim = torch.tensor(0.0, device=pred.device)
    
    return lambda_l1 * l1 + lambda_ssim * ssim


# ============================================================
# ⚠️ CRITICAL: Verify setup before optimization
# ============================================================

print("🔍 Pre-optimization Diagnostics:")
print("\n📊 Target Images:")
sample_view = enhanced_views[0]
print(f"   Type: {type(sample_view)}")
print(f"   Mode: {sample_view.mode}")
print(f"   Size: {sample_view.size}")
img_array = np.array(sample_view)
print(f"   Array shape: {img_array.shape}")
print(f"   Value range: [{img_array.min()}, {img_array.max()}]")
print(f"   Mean value: {img_array.mean():.1f}")

# Check Gaussian initialization
with torch.no_grad():
    xyz = model.xyz.cpu().numpy()
    colors = model.get_colors().cpu().numpy()
    opacity = model.opacity.cpu().numpy()
    scales = model.scales.cpu().numpy()
    
print(f"\n📊 Initial Gaussians:")
print(f"   Points: {len(xyz):,}")
print(f"   XYZ center: ({xyz.mean(0)[0]:.3f}, {xyz.mean(0)[1]:.3f}, {xyz.mean(0)[2]:.3f})")
print(f"   XYZ extent: {xyz.max() - xyz.min():.3f}")
print(f"   Opacity: [{opacity.min():.4f}, {opacity.max():.4f}] (mean={opacity.mean():.4f})")
print(f"   Scales: [{scales.min():.6f}, {scales.max():.6f}] (mean={scales.mean():.6f})")
print(f"   Colors: [{colors.min():.4f}, {colors.max():.4f}] (mean={colors.mean():.4f})")

print(f"\n📊 Camera Setup:")
print(f"   Radius: {RADIUS}")
print(f"   Image size: {IMAGE_SIZE}")
print(f"   Num views: {len(camera_poses)}")

# ============================================================
# Prepare target images as tensors
# ============================================================
target_tensors = []
for img in enhanced_views:
    img_resized = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
    # ⚠️ FIX: Ensure RGB (3 channels), not RGBA
    if img_resized.mode == 'RGBA':
        img_resized = img_resized.convert('RGB')
    img_tensor = torch.tensor(np.array(img_resized) / 255.0, dtype=torch.float32, device=device)
    # Ensure shape is [H, W, 3]
    if img_tensor.dim() == 3 and img_tensor.shape[-1] == 3:
        target_tensors.append(img_tensor)
    else:
        print(f"⚠️ Warning: Unexpected tensor shape {img_tensor.shape}")
        target_tensors.append(img_tensor[..., :3])

# Prepare depth tensors if available
depth_tensors = []
if 'depth_maps' in dir() and depth_maps:
    for depth in depth_maps:
        # Resize depth to match image size
        depth_pil = Image.fromarray((depth * 255).astype(np.uint8))
        depth_resized = depth_pil.resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)
        depth_tensor = torch.tensor(np.array(depth_resized) / 255.0, dtype=torch.float32, device=device)
        depth_tensors.append(depth_tensor)
    print(f"✅ Prepared {len(depth_tensors)} depth maps for depth-guided optimization")

print(f"\n✅ Prepared {len(target_tensors)} target images at {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"   Target tensor shape: {target_tensors[0].shape}")
print(f"   Target value range: [{target_tensors[0].min():.3f}, {target_tensors[0].max():.3f}]")

# ============================================================
# ⚠️ TEST RENDER: Check if Gaussians are visible BEFORE training
# ============================================================
print("\n🔍 Test render BEFORE optimization...")
with torch.no_grad():
    test_render, test_alpha = render_gaussians(model, camera_poses[0], intrinsics, IMAGE_SIZE)
    test_np = test_render.detach().cpu().numpy()
    print(f"   Rendered shape: {test_np.shape}")
    print(f"   Rendered range: [{test_np.min():.4f}, {test_np.max():.4f}]")
    print(f"   Rendered mean: {test_np.mean():.4f}")
    
    if test_np.max() < 0.01:
        print("\n   ⚠️ WARNING: Initial render is nearly BLACK!")
        print("   This means cameras are not seeing the Gaussians.")
        print("   Possible causes:")
        print("     1. Gaussians are too small (scales)")
        print("     2. Gaussians are invisible (opacity)")
        print("     3. Camera is not pointing at Gaussians")
        
        # Try to fix by adjusting scale
        print("\n   🔧 Attempting to fix: Increasing initial scales...")
        model.scales_raw.data += 1.0  # Increase scale by e^1 ≈ 2.7x
        
        # Re-test
        test_render2, _ = render_gaussians(model, camera_poses[0], intrinsics, IMAGE_SIZE)
        test_np2 = test_render2.detach().cpu().numpy()
        print(f"   After scale fix - render mean: {test_np2.mean():.4f}")
        
        if test_np2.max() < 0.01:
            print("   ⚠️ Still black - increasing opacity...")
            model.opacity_raw.data += 2.0  # Higher starting opacity
            
            test_render3, _ = render_gaussians(model, camera_poses[0], intrinsics, IMAGE_SIZE)
            test_np3 = test_render3.detach().cpu().numpy()
            print(f"   After opacity fix - render mean: {test_np3.mean():.4f}")
    else:
        print("   ✅ Initial render looks good!")

# Show test render vs target
fig, axes = plt.subplots(1, 3 if depth_tensors else 2, figsize=(15, 5))
axes[0].imshow(test_render.detach().cpu().numpy())
axes[0].set_title(f'Initial Render (mean={test_render.mean():.3f})')
axes[0].axis('off')
axes[1].imshow(target_tensors[0].cpu().numpy())
axes[1].set_title(f'Target (mean={target_tensors[0].mean():.3f})')
axes[1].axis('off')
if depth_tensors:
    axes[2].imshow(depth_tensors[0].cpu().numpy(), cmap='magma')
    axes[2].set_title('Depth Map')
    axes[2].axis('off')
plt.tight_layout()
plt.savefig(DIRS['gsplat'] / "render_vs_target.png")
plt.show()

# ============================================================
# Optimizer with CONSERVATIVE learning rates and better scheduling
# ============================================================
# ⚠️ Key insight: Start with smaller LRs to prevent divergence

# Different learning rates for different parameter groups
lr_config = {
    'xyz': 1e-5,           # Very small - positions are already good from mesh
    'f_dc': 2e-3,          # Color is important - needs faster learning
    'f_rest': 1e-4,        # SH coefficients - low priority
    'opacity': 5e-3,       # Moderate - visibility matters
    'scales': 5e-4,        # Small - scales are sensitive
    'rotations': 1e-4,     # Small - rotations are sensitive
}

optimizer = torch.optim.Adam([
    {'params': model.xyz, 'lr': lr_config['xyz'], 'name': 'xyz'},
    {'params': model.f_dc, 'lr': lr_config['f_dc'], 'name': 'f_dc'},
    {'params': model.f_rest, 'lr': lr_config['f_rest'], 'name': 'f_rest'},
    {'params': model.opacity_raw, 'lr': lr_config['opacity'], 'name': 'opacity'},
    {'params': model.scales_raw, 'lr': lr_config['scales'], 'name': 'scales'},
    {'params': model.rotations, 'lr': lr_config['rotations'], 'name': 'rotations'},
])

# Cosine annealing with warm restarts for better convergence
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=200, T_mult=2, eta_min=1e-6
)

# ============================================================
# Training loop with better monitoring and depth guidance
# ============================================================
NUM_ITERATIONS = 1500  # More iterations for better convergence
USE_DEPTH_GUIDANCE = len(depth_tensors) > 0
DEPTH_WEIGHT = 0.1  # Weight for depth-guided loss

losses = []
best_loss = float('inf')
best_state = None
gradient_norms = []

print(f"\n🚀 Starting optimization for {NUM_ITERATIONS} iterations...")
print(f"   Depth guidance: {'Enabled' if USE_DEPTH_GUIDANCE else 'Disabled'}")
pbar = tqdm(range(NUM_ITERATIONS))

for iteration in pbar:
    optimizer.zero_grad()
    
    # Sample random view (with bias towards front views in early iterations)
    if iteration < 300:
        # Early: focus on frontal views
        view_idx = np.random.choice([0, 1, 2, 14, 15], p=[0.3, 0.2, 0.15, 0.2, 0.15])
    else:
        # Later: sample all views uniformly
        view_idx = np.random.randint(0, 16)
    
    w2c = camera_poses[view_idx]
    target = target_tensors[view_idx]
    
    # Render
    rendered, alpha = render_gaussians(model, w2c, intrinsics, IMAGE_SIZE)
    
    # ⚠️ Check for NaN/Inf early
    if torch.isnan(rendered).any() or torch.isinf(rendered).any():
        print(f"\n⚠️ NaN/Inf detected at iteration {iteration}! Stopping...")
        break
    
    # Combined L1 + SSIM loss for photometric quality
    photo_loss = combined_loss(rendered, target, lambda_l1=0.8, lambda_ssim=0.2)
    
    # Depth guidance loss (optional)
    depth_loss = torch.tensor(0.0, device=device)
    if USE_DEPTH_GUIDANCE and depth_tensors:
        depth_target = depth_tensors[view_idx]
        # Compare alpha map (proxy for depth) with estimated depth
        # Higher alpha = more opaque = should correlate with closer objects
        alpha_squeezed = alpha.squeeze() if alpha.dim() > 2 else alpha
        if alpha_squeezed.shape == depth_target.shape:
            # Normalize both to [0, 1]
            alpha_norm = (alpha_squeezed - alpha_squeezed.min()) / (alpha_squeezed.max() - alpha_squeezed.min() + 1e-8)
            depth_loss = F.mse_loss(alpha_norm, depth_target)
    
    # Total loss
    loss = photo_loss + DEPTH_WEIGHT * depth_loss
    
    loss.backward()
    
    # ⚠️ Check gradients
    grad_norm = 0
    for p in model.parameters():
        if p.grad is not None:
            grad_norm += p.grad.norm().item() ** 2
    grad_norm = grad_norm ** 0.5
    gradient_norms.append(grad_norm)
    
    # Skip update if gradients are extreme
    if grad_norm > 50:
        if iteration % 100 == 0:
            print(f"\n⚠️ Large gradient ({grad_norm:.1f}) at iter {iteration}, clipping...")
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    # Standard gradient clipping for stability
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    optimizer.step()
    scheduler.step()
    
    losses.append(loss.item())
    
    # Save best model
    if loss.item() < best_loss:
        best_loss = loss.item()
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    
    # Progress update
    if iteration % 100 == 0:
        with torch.no_grad():
            render_mean = rendered.mean().item()
        current_lr = optimizer.param_groups[0]['lr']
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}', 
            'best': f'{best_loss:.4f}',
            'view': view_idx,
            'render': f'{render_mean:.3f}',
            'grad': f'{grad_norm:.2f}',
            'lr': f'{current_lr:.2e}'
        })

# Restore best model if training diverged
if len(losses) > 100 and losses[-1] > losses[100] * 1.5:  # Final loss > 150% of loss at iter 100
    print(f"\n⚠️ Training may have diverged (loss went up). Restoring best model...")
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"   Restored to best loss: {best_loss:.4f}")

print(f"\n✅ Optimization complete!")
print(f"   Initial loss: {losses[0]:.4f}")
print(f"   Best loss: {best_loss:.4f}")
print(f"   Final loss: {losses[-1]:.4f}")
print(f"   Improvement: {(1 - losses[-1]/losses[0])*100:.1f}%")

# Test render after optimization
with torch.no_grad():
    final_render, _ = render_gaussians(model, camera_poses[0], intrinsics, IMAGE_SIZE)
    print(f"   Final render mean: {final_render.mean():.4f}")

# Plot loss curve and gradient norms
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(losses, alpha=0.7)
axes[0].axhline(y=best_loss, color='g', linestyle='--', label=f'Best: {best_loss:.4f}')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss')
axes[0].set_title('gsplat Optimization Loss')
axes[0].legend()
axes[0].grid(True)

# Smooth gradient norms for visualization
window = 50
smoothed_grads = np.convolve(gradient_norms, np.ones(window)/window, mode='valid')
axes[1].plot(smoothed_grads, alpha=0.7)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Gradient Norm (smoothed)')
axes[1].set_title('Gradient Stability')
axes[1].grid(True)

plt.tight_layout()
plt.savefig(DIRS['gsplat'] / "optimization_curves.png")
plt.show()

In [ ]:
# ============================================================
# Save Optimized Model to PLY
# ============================================================

def save_gaussian_ply(model, output_path, verbose=True):
    """
    Save Gaussian model to PLY file in gsplat-compatible format.
    
    ⚠️ CRITICAL: This saves the RAW parameters (before activation functions)
    so the model can be loaded and rendered correctly later.
    """
    with torch.no_grad():
        # Get raw parameters (before activation)
        xyz = model.xyz.cpu().numpy()
        colors = model.f_dc.cpu().numpy()  # Raw SH DC coefficients
        f_rest = model.f_rest.cpu().numpy()
        opacity_raw = model.opacity_raw.cpu().numpy()  # Raw (before sigmoid)
        scales_raw = model.scales_raw.cpu().numpy()    # Raw (before exp)
        rotations = model.rotations.cpu().numpy()      # Already normalized in forward()
        
        # Normalize rotations for storage
        rot_norms = np.linalg.norm(rotations, axis=1, keepdims=True)
        rotations = rotations / (rot_norms + 1e-8)
    
    num_points = len(xyz)
    
    if verbose:
        print(f"\n💾 Saving Gaussian model to PLY...")
        print(f"   Points: {num_points:,}")
        print(f"   XYZ range: [{xyz.min():.4f}, {xyz.max():.4f}]")
        
        # Show activated values for debugging
        opacity_activated = 1 / (1 + np.exp(-opacity_raw))
        scales_activated = np.exp(np.clip(scales_raw, -10, 5))
        print(f"   Opacity (activated): [{opacity_activated.min():.4f}, {opacity_activated.max():.4f}]")
        print(f"   Scales (activated): [{scales_activated.min():.6f}, {scales_activated.max():.6f}]")
    
    # Build PLY data structure
    dtype_full = [
        ('x', 'f4'), ('y', 'f4'), ('z', 'f4'),
        ('f_dc_0', 'f4'), ('f_dc_1', 'f4'), ('f_dc_2', 'f4'),
    ]
    for i in range(f_rest.shape[1]):
        dtype_full.append((f'f_rest_{i}', 'f4'))
    dtype_full.extend([
        ('opacity', 'f4'),
        ('scale_0', 'f4'), ('scale_1', 'f4'), ('scale_2', 'f4'),
        ('rot_0', 'f4'), ('rot_1', 'f4'), ('rot_2', 'f4'), ('rot_3', 'f4'),
    ])
    
    elements = np.zeros(num_points, dtype=dtype_full)
    elements['x'] = xyz[:, 0]
    elements['y'] = xyz[:, 1]
    elements['z'] = xyz[:, 2]
    elements['f_dc_0'] = colors[:, 0]
    elements['f_dc_1'] = colors[:, 1]
    elements['f_dc_2'] = colors[:, 2]
    for i in range(f_rest.shape[1]):
        elements[f'f_rest_{i}'] = f_rest[:, i]
    elements['opacity'] = opacity_raw  # Store raw values
    elements['scale_0'] = scales_raw[:, 0]
    elements['scale_1'] = scales_raw[:, 1]
    elements['scale_2'] = scales_raw[:, 2]
    elements['rot_0'] = rotations[:, 0]
    elements['rot_1'] = rotations[:, 1]
    elements['rot_2'] = rotations[:, 2]
    elements['rot_3'] = rotations[:, 3]
    
    el = PlyElement.describe(elements, 'vertex')
    PlyData([el]).write(str(output_path))
    
    if verbose:
        file_size_mb = Path(output_path).stat().st_size / 1024 / 1024
        print(f"✅ Saved: {output_path}")
        print(f"   File size: {file_size_mb:.2f} MB")
    
    return {
        'num_points': num_points,
        'file_size_mb': Path(output_path).stat().st_size / 1024 / 1024
    }


# Save optimized model
OPTIMIZED_PLY_PATH = DIRS['gsplat'] / "optimized_gaussian.ply"
save_stats = save_gaussian_ply(model, str(OPTIMIZED_PLY_PATH), verbose=True)

# ============================================================
# ⚠️ CRITICAL DIAGNOSTICS: Verify model quality
# ============================================================
print("\n🔍 Post-Optimization Model Diagnostics:")

with torch.no_grad():
    params = model()
    opacity_values = params['opacity'].cpu().numpy()
    scale_values = params['scales'].cpu().numpy()
    color_values = params['colors'].cpu().numpy()
    
    print(f"\n📊 Opacity (sigmoid(opacity_raw)):")
    print(f"   Range: [{opacity_values.min():.6f}, {opacity_values.max():.6f}]")
    print(f"   Mean: {opacity_values.mean():.6f}")
    print(f"   Median: {np.median(opacity_values):.6f}")
    print(f"   % > 0.5: {(opacity_values > 0.5).sum() / len(opacity_values) * 100:.1f}%")
    print(f"   % > 0.9: {(opacity_values > 0.9).sum() / len(opacity_values) * 100:.1f}%")
    
    print(f"\n📊 Scales (exp(scales_raw)):")
    print(f"   Range: [{scale_values.min():.6f}, {scale_values.max():.6f}]")
    print(f"   Mean: {scale_values.mean():.6f}")
    print(f"   Median: {np.median(scale_values):.6f}")
    
    print(f"\n📊 Colors (0.5 + C0 * f_dc):")
    print(f"   Range: [{color_values.min():.6f}, {color_values.max():.6f}]")
    print(f"   Mean: {color_values.mean():.6f}")
    print(f"   % in valid range [0,1]: {((color_values >= 0) & (color_values <= 1)).sum() / color_values.size * 100:.1f}%")

# ============================================================
# Quality Check: Render validation images
# ============================================================
print("\n📸 Rendering validation images...")

validation_renders = []
with torch.no_grad():
    for i in [0, 4, 8, 12]:
        rendered, _ = render_gaussians(model, camera_poses[i], intrinsics, IMAGE_SIZE)
        rendered_np = (rendered.cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
        validation_renders.append(rendered_np)
        
        # Save individual render
        Image.fromarray(rendered_np).save(DIRS['gsplat'] / f"final_render_view{i}.png")

print(f"✅ Saved 4 validation renders to {DIRS['gsplat']}")

# Check if renders are reasonable
render_brightness = [r.mean() for r in validation_renders]
print(f"\n📊 Render brightness: {render_brightness}")

if min(render_brightness) < 10:
    print("⚠️ WARNING: Some renders are very dark!")
    print("   This may indicate optimization issues.")
elif max(render_brightness) > 245:
    print("⚠️ WARNING: Some renders are overexposed!")
else:
    print("✅ Render brightness looks reasonable!")

# Stage 6: MVCRM - Multi-View Consistency Refinement

**Input:** Optimized Gaussian Splats + Enhanced Views + Depth Maps  
**Output:** Refined Gaussian Splats with multi-view consistency  
**Time:** ~2 minutes

MVCRM iteratively refines the 3D model by:
1. Comparing rendered views with enhanced target views
2. Back-projecting differences to update Gaussian parameters
3. Using depth maps to ensure geometric consistency
4. Applying smoothing to prevent artifacts

In [ ]:
print("\n" + "="*60)
print("🔄 STAGE 6: MVCRM - Multi-View Consistency Refinement")
print("="*60)

# ============================================================
# MVCRM Configuration
# ============================================================

class MVCRMConfig:
    """Configuration for MVCRM refinement."""
    max_iterations: int = 3  # Number of refinement passes
    views_per_iteration: int = 8  # Views to process per iteration
    learning_rate: float = 0.02  # Refinement learning rate
    lr_decay: float = 0.7  # LR decay per iteration
    depth_weight: float = 0.2  # Weight for depth consistency
    color_threshold: float = 0.1  # Minimum color difference to update
    use_depth_guidance: bool = True


config = MVCRMConfig()
print(f"\n📋 MVCRM Configuration:")
print(f"   Iterations: {config.max_iterations}")
print(f"   Views per iteration: {config.views_per_iteration}")
print(f"   Learning rate: {config.learning_rate}")
print(f"   Depth guidance: {config.use_depth_guidance}")

# ============================================================
# MVCRM Core Functions
# ============================================================

def compute_view_difference(rendered, target, mask=None):
    """
    Compute pixel-wise difference between rendered and target views.
    
    Returns:
        diff: (H, W, 3) signed difference
        diff_magnitude: (H, W) magnitude of difference
    """
    diff = target - rendered
    diff_magnitude = torch.norm(diff, dim=-1)
    
    if mask is not None:
        diff = diff * mask.unsqueeze(-1)
        diff_magnitude = diff_magnitude * mask
    
    return diff, diff_magnitude


def depth_guided_weight(depth_map, depth_rendered, threshold=0.2):
    """
    Create weight mask based on depth consistency.
    
    Areas where depths are consistent get higher weights.
    """
    if depth_map is None or depth_rendered is None:
        return torch.ones_like(depth_rendered)
    
    # Normalize both to [0, 1]
    depth_norm = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min() + 1e-8)
    rendered_norm = (depth_rendered - depth_rendered.min()) / (depth_rendered.max() - depth_rendered.min() + 1e-8)
    
    # Compute consistency score
    depth_diff = torch.abs(depth_norm - rendered_norm)
    weight = torch.exp(-depth_diff / threshold)
    
    return weight


def mvcrm_refine_step(model, target, depth_target, camera_pose, intrinsics, 
                       image_size, lr=0.01, depth_weight=0.2):
    """
    Single MVCRM refinement step for one view.
    
    This back-projects the difference between rendered and target views
    to update Gaussian parameters with depth-aware weighting.
    """
    # Create optimizer for this step
    optimizer = torch.optim.Adam([
        {'params': model.f_dc, 'lr': lr * 2.0},  # Color updates
        {'params': model.opacity_raw, 'lr': lr * 0.5},
    ], lr=lr)
    
    # Render current state
    rendered, alpha = render_gaussians(model, camera_pose, intrinsics, image_size)
    
    # Compute photometric loss
    photo_loss = F.l1_loss(rendered, target)
    
    # Compute depth-guided loss if available
    depth_loss = torch.tensor(0.0, device=device)
    if depth_target is not None and config.use_depth_guidance:
        # Use alpha as proxy for rendered depth
        alpha_squeezed = alpha.squeeze() if alpha.dim() > 2 else alpha
        if alpha_squeezed.shape == depth_target.shape:
            weight_mask = depth_guided_weight(depth_target, alpha_squeezed)
            weighted_diff = (rendered - target) * weight_mask.unsqueeze(-1)
            depth_loss = weighted_diff.abs().mean()
    
    # Total loss
    loss = photo_loss + depth_weight * depth_loss
    
    # Backward and update
    optimizer.zero_grad()
    loss.backward()
    
    # Clip gradients
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
    
    optimizer.step()
    
    return loss.item(), photo_loss.item()


# ============================================================
# Run MVCRM Refinement Loop
# ============================================================

print("\n🚀 Starting MVCRM refinement...")

mvcrm_history = []
initial_losses = []

# Compute initial losses for all views
print("\n📊 Initial View Losses:")
with torch.no_grad():
    for i in range(len(camera_poses)):
        rendered, _ = render_gaussians(model, camera_poses[i], intrinsics, IMAGE_SIZE)
        loss = F.l1_loss(rendered, target_tensors[i]).item()
        initial_losses.append(loss)
        if i < 4:
            print(f"   View {i}: {loss:.4f}")

print(f"   Average initial loss: {np.mean(initial_losses):.4f}")

# MVCRM iterations
for iteration in range(config.max_iterations):
    print(f"\n--- MVCRM Iteration {iteration + 1}/{config.max_iterations} ---")
    
    current_lr = config.learning_rate * (config.lr_decay ** iteration)
    print(f"   Learning rate: {current_lr:.4f}")
    
    iteration_losses = []
    
    # Select views for this iteration (prioritize high-error views)
    with torch.no_grad():
        view_losses = []
        for i in range(len(camera_poses)):
            rendered, _ = render_gaussians(model, camera_poses[i], intrinsics, IMAGE_SIZE)
            loss = F.l1_loss(rendered, target_tensors[i]).item()
            view_losses.append((i, loss))
    
    # Sort by loss and take top views
    view_losses.sort(key=lambda x: x[1], reverse=True)
    selected_views = [v[0] for v in view_losses[:config.views_per_iteration]]
    print(f"   Selected views (highest error): {selected_views}")
    
    # Refine each selected view
    for view_idx in tqdm(selected_views, desc=f"Iteration {iteration+1}"):
        target = target_tensors[view_idx]
        depth_target = depth_tensors[view_idx] if depth_tensors else None
        camera_pose = camera_poses[view_idx]
        
        loss, photo_loss = mvcrm_refine_step(
            model, target, depth_target, camera_pose, intrinsics,
            IMAGE_SIZE, lr=current_lr, depth_weight=config.depth_weight
        )
        iteration_losses.append(loss)
    
    avg_loss = np.mean(iteration_losses)
    print(f"   Average loss this iteration: {avg_loss:.4f}")
    mvcrm_history.append(avg_loss)

# Compute final losses
print("\n📊 Final View Losses:")
final_losses = []
with torch.no_grad():
    for i in range(len(camera_poses)):
        rendered, _ = render_gaussians(model, camera_poses[i], intrinsics, IMAGE_SIZE)
        loss = F.l1_loss(rendered, target_tensors[i]).item()
        final_losses.append(loss)
        if i < 4:
            improvement = (initial_losses[i] - loss) / initial_losses[i] * 100
            print(f"   View {i}: {loss:.4f} (improved {improvement:.1f}%)")

print(f"\n✅ MVCRM Refinement Complete!")
print(f"   Average initial loss: {np.mean(initial_losses):.4f}")
print(f"   Average final loss: {np.mean(final_losses):.4f}")
print(f"   Overall improvement: {(1 - np.mean(final_losses)/np.mean(initial_losses))*100:.1f}%")

# Visualize MVCRM results
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Top row: Before (initial renders after gsplat)
# Bottom row: After MVCRM

with torch.no_grad():
    for i in range(4):
        view_idx = i * 4  # Views 0, 4, 8, 12
        
        # Render after MVCRM
        rendered, _ = render_gaussians(model, camera_poses[view_idx], intrinsics, IMAGE_SIZE)
        rendered_np = rendered.cpu().numpy()
        
        # Target
        target_np = target_tensors[view_idx].cpu().numpy()
        
        # Show target on top row
        axes[0, i].imshow(target_np)
        axes[0, i].set_title(f"Target View {view_idx}")
        axes[0, i].axis('off')
        
        # Show rendered on bottom row
        axes[1, i].imshow(rendered_np)
        loss = final_losses[view_idx]
        axes[1, i].set_title(f"Rendered (loss={loss:.4f})")
        axes[1, i].axis('off')

plt.suptitle("MVCRM Refinement Results: Target (top) vs Rendered (bottom)", fontsize=14)
plt.tight_layout()
plt.savefig(DIRS['mvcrm'] / "mvcrm_comparison.png", dpi=150)
plt.show()

# Stage 6: Generate Final Outputs

In [ ]:
# ============================================================
# 🏆 FINAL OUTPUT GENERATION
# ============================================================

print("\n" + "="*60)
print("🏆 FINAL OUTPUT GENERATION")
print("="*60)

import imageio
import math
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm

# ============================================================
# Analyze final Gaussian model for optimal camera settings
# ============================================================
with torch.no_grad():
    xyz = model.xyz.cpu().numpy()
    
print(f"\n📊 Final Gaussian Point Cloud:")
print(f"   Points: {len(xyz):,}")
print(f"   X range: [{xyz[:, 0].min():.3f}, {xyz[:, 0].max():.3f}]")
print(f"   Y range: [{xyz[:, 1].min():.3f}, {xyz[:, 1].max():.3f}]")
print(f"   Z range: [{xyz[:, 2].min():.3f}, {xyz[:, 2].max():.3f}]")

# Calculate bounding box and optimal camera settings
bbox_min = xyz.min(axis=0)
bbox_max = xyz.max(axis=0)
center = (bbox_min + bbox_max) / 2
extent = (bbox_max - bbox_min).max()

print(f"   Center: ({center[0]:.3f}, {center[1]:.3f}, {center[2]:.3f})")
print(f"   Extent: {extent:.3f}")

# Auto-calculate optimal camera radius
VIDEO_RADIUS = max(extent * 2.5, 0.8)
print(f"   Video camera radius: {VIDEO_RADIUS:.3f}")

# ============================================================
# Video rendering functions
# ============================================================

VIDEO_SIZE = 512  # Higher resolution for final video

def get_intrinsics_video(fov_deg=49.1, image_size=512):
    """Camera intrinsics for video rendering."""
    fov_rad = math.radians(fov_deg)
    focal = image_size / (2 * math.tan(fov_rad / 2))
    K = np.array([
        [focal, 0, image_size / 2],
        [0, focal, image_size / 2],
        [0, 0, 1]
    ], dtype=np.float32)
    return K


def create_video_camera_pose(elevation_deg, azimuth_deg, radius, center):
    """Create camera pose for video rendering."""
    elev = math.radians(elevation_deg)
    azim = math.radians(azimuth_deg)
    
    # Camera position (Y-up convention)
    x = center[0] + radius * math.cos(elev) * math.sin(azim)
    y = center[1] + radius * math.sin(elev)
    z = center[2] + radius * math.cos(elev) * math.cos(azim)
    
    cam_pos = np.array([x, y, z])
    look_at = np.array(center)
    up = np.array([0.0, 1.0, 0.0])
    
    # Camera basis
    forward = look_at - cam_pos
    forward_norm = np.linalg.norm(forward)
    if forward_norm < 1e-6:
        forward = np.array([0.0, 0.0, -1.0])
    else:
        forward = forward / forward_norm
    
    right = np.cross(forward, up)
    right_norm = np.linalg.norm(right)
    if right_norm < 1e-6:
        up = np.array([0.0, 0.0, 1.0])
        right = np.cross(forward, up)
        right_norm = np.linalg.norm(right)
    right = right / (right_norm + 1e-8)
    up_new = np.cross(right, forward)
    
    # World-to-camera matrix
    w2c = np.eye(4, dtype=np.float32)
    w2c[0, :3] = right
    w2c[1, :3] = up_new
    w2c[2, :3] = -forward
    w2c[:3, 3] = -w2c[:3, :3] @ cam_pos
    
    return w2c


video_intrinsics = get_intrinsics_video(fov_deg=49.1, image_size=VIDEO_SIZE)

# ============================================================
# Test render before generating full video
# ============================================================
print("\n🔍 Testing video render quality...")

test_w2c = create_video_camera_pose(20.0, 0.0, VIDEO_RADIUS, center)
with torch.no_grad():
    # Need to redefine render for video size
    params = model()
    viewmat = torch.tensor(test_w2c, dtype=torch.float32, device=device)
    K_tensor = torch.tensor(video_intrinsics, dtype=torch.float32, device=device)
    
    test_rgb, test_alpha, _ = rasterization(
        means=params['xyz'],
        quats=params['rotations'],
        scales=params['scales'],
        opacities=params['opacity'],
        colors=params['colors'],
        viewmats=viewmat.unsqueeze(0),
        Ks=K_tensor.unsqueeze(0),
        width=VIDEO_SIZE,
        height=VIDEO_SIZE,
        packed=False,
        render_mode="RGB",
    )
    test_rgb = test_rgb[0]
    test_rgb_np = test_rgb.cpu().numpy()

print(f"   Test frame brightness: mean={test_rgb_np.mean():.3f}, max={test_rgb_np.max():.3f}")

# Adaptive radius adjustment if needed
if test_rgb_np.max() < 0.05:
    print("   ⚠️ Frame too dark, adjusting camera...")
    for mult in [1.5, 2.0, 3.0]:
        VIDEO_RADIUS = extent * mult
        test_w2c = create_video_camera_pose(20.0, 0.0, VIDEO_RADIUS, center)
        viewmat = torch.tensor(test_w2c, dtype=torch.float32, device=device)
        
        with torch.no_grad():
            test_rgb2, _, _ = rasterization(
                means=params['xyz'],
                quats=params['rotations'],
                scales=params['scales'],
                opacities=params['opacity'],
                colors=params['colors'],
                viewmats=viewmat.unsqueeze(0),
                Ks=K_tensor.unsqueeze(0),
                width=VIDEO_SIZE,
                height=VIDEO_SIZE,
                packed=False,
                render_mode="RGB",
            )
            if test_rgb2[0].max().item() > 0.05:
                print(f"   ✅ Found good radius: {VIDEO_RADIUS:.3f}")
                break
else:
    print("   ✅ Test frame looks good!")

# ============================================================
# Generate 360° turntable video
# ============================================================
print(f"\n🎬 Rendering 360° turntable video at {VIDEO_SIZE}x{VIDEO_SIZE}...")

NUM_FRAMES = 120
video_frames = []

with torch.no_grad():
    params = model()
    K_tensor = torch.tensor(video_intrinsics, dtype=torch.float32, device=device)
    
    for frame_idx, azim in enumerate(tqdm(np.linspace(0, 360, NUM_FRAMES, endpoint=False))):
        # Create camera pose
        w2c = create_video_camera_pose(20.0, azim, VIDEO_RADIUS, center)
        viewmat = torch.tensor(w2c, dtype=torch.float32, device=device)
        
        # Render
        rgb, alpha, _ = rasterization(
            means=params['xyz'],
            quats=params['rotations'],
            scales=params['scales'],
            opacities=params['opacity'],
            colors=params['colors'],
            viewmats=viewmat.unsqueeze(0),
            Ks=K_tensor.unsqueeze(0),
            width=VIDEO_SIZE,
            height=VIDEO_SIZE,
            packed=False,
            render_mode="RGB",
        )
        
        # Convert to uint8
        frame = (rgb[0].cpu().numpy().clip(0, 1) * 255).astype(np.uint8)
        video_frames.append(frame)

# Quality check
frame_brightness = [f.mean() for f in video_frames[::10]]
print(f"\n📊 Video Quality Check:")
print(f"   Frame brightness: min={min(frame_brightness):.1f}, max={max(frame_brightness):.1f}, avg={np.mean(frame_brightness):.1f}")

if min(frame_brightness) < 5:
    print("   ⚠️ Some frames are very dark!")
elif np.mean(frame_brightness) < 30:
    print("   ⚠️ Video is dim overall")
else:
    print("   ✅ Video brightness looks good!")

# Save video
video_path = DIRS['output'] / "glimpse3d_360.mp4"
imageio.mimsave(str(video_path), video_frames, fps=30, quality=8)
print(f"\n✅ Video saved: {video_path}")

# Save debug frames
for i, frame_idx in enumerate([0, 30, 60, 90]):
    if frame_idx < len(video_frames):
        debug_path = DIRS['output'] / f"frame_{frame_idx:03d}.png"
        Image.fromarray(video_frames[frame_idx]).save(debug_path)
print(f"✅ Debug frames saved to {DIRS['output']}")

# ============================================================
# Save pipeline diagnostics
# ============================================================
diagnostics = {
    'gaussian_count': len(xyz),
    'model_center': center.tolist(),
    'model_extent': float(extent),
    'camera_radius': float(VIDEO_RADIUS),
    'video_resolution': VIDEO_SIZE,
    'num_frames': NUM_FRAMES,
    'avg_brightness': float(np.mean(frame_brightness)),
    'optimization_iterations': NUM_ITERATIONS,
    'mvcrm_iterations': config.max_iterations,
    'depth_guidance': config.use_depth_guidance,
}

import json
diag_path = DIRS['output'] / "pipeline_diagnostics.json"
with open(diag_path, 'w') as f:
    json.dump(diagnostics, f, indent=2)
print(f"✅ Diagnostics saved: {diag_path}")

In [ ]:
# Copy final files
import shutil

# Copy optimized PLY
final_ply = DIRS['output'] / "final_gaussian.ply"
shutil.copy(OPTIMIZED_PLY_PATH, final_ply)

# Copy mesh
shutil.copy(DIRS['triposr'] / "mesh.glb", DIRS['output'] / "initial_mesh.glb")
shutil.copy(DIRS['triposr'] / "mesh.obj", DIRS['output'] / "initial_mesh.obj")

# Copy best views
for i in [0, 4, 8, 12]:
    shutil.copy(
        DIRS['syncdreamer'] / f"view_{i:02d}_e{int(ELEVATIONS[i])}_a{int(AZIMUTHS[i])}.png",
        DIRS['output'] / f"view_{i:02d}.png"
    )

print("\n📁 Final output files:")
for f in sorted(DIRS['output'].iterdir()):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name} ({size_mb:.1f} MB)")

In [ ]:
# Display video
from IPython.display import HTML
from base64 import b64encode

mp4 = open(video_path, 'rb').read()
data_url = f"data:video/mp4;base64,{b64encode(mp4).decode()}"
HTML(f'''
<h3>🏆 Glimpse3D Result</h3>
<video width="600" controls autoplay loop>
    <source src="{data_url}" type="video/mp4">
</video>
''')

# 📥 Download All Results

In [ ]:
from google.colab import files

# Create final ZIP
output_zip = str(WORK_DIR / "glimpse3d_complete_output")
shutil.make_archive(output_zip, 'zip', DIRS['output'])

print("📥 Downloading Glimpse3D results...")
files.download(f"{output_zip}.zip")

print("\n" + "="*60)
print("✅ GLIMPSE3D PIPELINE COMPLETE!")
print("="*60)
print(f"\nDownloaded: glimpse3d_complete_output.zip")
print("\nContents:")
print("  - final_gaussian.ply   : Optimized Gaussian Splats")
print("  - initial_mesh.glb/obj : TripoSR mesh")
print("  - glimpse3d_360.mp4    : 360° turntable video")
print("  - view_*.png           : Multi-view images")

---

## 🎉 Pipeline Complete!

You now have:
1. **final_gaussian.ply** - View in any Gaussian Splat viewer
2. **initial_mesh.glb** - View in 3D viewers like Blender, online GLB viewers
3. **glimpse3d_360.mp4** - Share as video

### Recommended Viewers
- **Gaussian Splats**: [SuperSplat](https://playcanvas.com/supersplat/editor), [Luma AI Viewer](https://lumalabs.ai/)
- **GLB Mesh**: [glTF Viewer](https://gltf-viewer.donmccurdy.com/), Blender

### Tips for Better Results
1. Use high-quality input images with clean backgrounds
2. Objects should be centered and fill ~80% of the frame
3. Avoid reflective or transparent surfaces
4. Run more gsplat iterations (2000+) for higher quality